[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wdittaya/AI-for-digital-health-course/blob/main/lab06_speech_processing.ipynb)

# ปฏิบัติการที่ 6 (Lab 6): การประมวลผลเสียงพูดสำหรับงานด้านสุขภาพ (Speech Processing for Healthcare)

**รายวิชา:** 3099704 AI for Digital Health (2026) · **วันที่เรียน:** วันพฤหัสบดีที่ 15 ตุลาคม 2026 · **เวลาโดยประมาณ:** 75 นาที

## วัตถุประสงค์การเรียนรู้

- ถอดความเสียงพูดทางการแพทย์ด้วยโมเดลรู้จำเสียงพูดแบบเปิด (OpenAI Whisper) ซึ่งรันบน Colab ได้โดยไม่มีค่าใช้จ่ายและไม่ต้องใช้ API key
- ฟังคลิปเสียงของผู้ป่วยจริง และเปรียบเทียบ Whisper ขนาดเล็กกับขนาดใหญ่ในด้านความแม่นยำและความเร็ว
- วัดคุณภาพการถอดความด้วยอัตราความผิดพลาดระดับคำ (Word Error Rate, WER) และอธิบายได้ว่าเหตุใดค่า WER ที่ต่ำจึงยังอาจมีความผิดพลาดที่เป็นอันตรายแฝงอยู่
- สร้างกระบวนการประมวลผล (pipeline) ขนาดเล็กสำหรับแปลงบันทึกเสียง (voice note) เป็นสรุปแบบมีโครงสร้าง

> **แนวทางการทำปฏิบัติการ** (งานเดี่ยว)
>
> - ✅ **ส่วนหลัก — สำหรับผู้เรียนทุกคน ไม่จำเป็นต้องมีพื้นฐานการเขียนโปรแกรม** ให้รันเซลล์ (cell) ตามลำดับจากบนลงล่าง (คลิกปุ่ม ▶ หรือกด **Shift+Enter**) ในแต่ละ ⭐ **แบบฝึกหัด (Exercise)** ผู้เรียนเพียงเปลี่ยนค่าหนึ่งค่าหรือเติมโค้ดสั้น ๆ หนึ่งบรรทัด และมีเซลล์ ✅ **เฉลย** อยู่ถัดไป ให้ทดลองทำด้วยตนเองก่อน หากไม่สามารถดำเนินการต่อได้ ให้รันเซลล์เฉลยแล้วทำส่วนถัดไป จากนั้นให้ตอบคำถาม ✍️ ในเซลล์ข้อความที่กำหนด **คำตอบดังกล่าวคือชิ้นงานที่ต้องส่ง** ควรใช้หูฟัง เนื่องจากการฟังคลิปเสียงเป็นส่วนหนึ่งของปฏิบัติการ
> - 🏆 **โจทย์ท้าทายบนตารางอันดับ — ไม่บังคับ สำหรับผู้เรียนที่เขียนโปรแกรมได้** ส่วนท้ายของปฏิบัติการเป็นการถอดความคลิปเสียงผู้ป่วย 30 คลิปและส่งผลไปยังตารางอันดับ (leaderboard) ของรายวิชา ส่วนนี้ไม่บังคับและไม่นับเป็นชิ้นงานที่ต้องส่ง
>
> หากพบปัญหา สามารถสอบถามผู้สอนผ่านช่องแชตของ Zoom

> **Runtime (แนะนำให้ใช้ GPU):** ก่อนรันเซลล์ใด ให้เลือก `Runtime → Change runtime type → T4 GPU → Save` GPU รุ่น T4 ซึ่งใช้ได้โดยไม่มีค่าใช้จ่ายจะทำให้ Whisper ทำงานได้เร็วขึ้นอย่างมาก ปฏิบัติการนี้สามารถรันบน CPU ได้เช่นกัน แต่จะช้ากว่า (การเปรียบเทียบขนาดโมเดลจะใช้เวลาหลายนาทีแทนที่จะเป็นไม่กี่วินาที)

> 🏆 ตารางอันดับซึ่งไม่บังคับนี้ให้คะแนนโดยอัตโนมัติบน [Hugging Face leaderboard](https://huggingface.co/spaces/wdittaya/aidh-leaderboard) ของรายวิชา ให้ระบุ `NAME` และ `JOIN_CODE` ในเซลล์การตั้งค่าด้านล่างเฉพาะกรณีที่ต้องการเข้าร่วม (ผู้สอนจะประกาศรหัสเข้าร่วมในการเรียนการสอนผ่าน Zoom) อย่างไรก็ตาม ผู้เรียนทุกคนต้องรันเซลล์การตั้งค่าก่อน


In [ ]:
# @title ⚙️ การตั้งค่า — โปรดรันเซลล์นี้ก่อน (NAME และ JOIN_CODE ใช้เฉพาะโจทย์ท้าทายบนตารางอันดับ ซึ่งไม่บังคับ)
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   เซลล์นี้เป็นการ "เตรียมความพร้อม" ก่อนเริ่มปฏิบัติการ โดยทำ 3 อย่าง ได้แก่
#   1) ติดตั้งเครื่องมือ (ไลบรารี) เพิ่มเติมที่จำเป็นต้องใช้ใน Google Colab
#   2) กำหนดค่าพื้นฐาน เช่น รหัสปฏิบัติการ ชื่อที่จะแสดง และรหัสเข้าร่วมตารางอันดับ
#   3) สร้าง "ฟังก์ชัน" (ชุดคำสั่งสำเร็จรูปที่เรียกใช้ซ้ำได้) 2 ตัว คือ
#        - lab_data(...)              : ดาวน์โหลดไฟล์ข้อมูลของรายวิชา
#        - submit_to_leaderboard(...) : ส่งผลการทำนายไปยังตารางอันดับ (ไม่บังคับ)
#   ผู้เรียนเพียงกดรันเซลล์นี้ 1 ครั้ง ไม่จำเป็นต้องแก้ไขโค้ด
#   (ยกเว้นต้องการเข้าร่วมตารางอันดับ ให้กรอก NAME และ JOIN_CODE ด้านล่าง)
# ============================================================================

# ติดตั้งไลบรารีเพิ่มเติมลงในเครื่อง Colab
#   - เครื่องหมาย ! ด้านหน้า หมายถึงสั่งงานระบบปฏิบัติการ (ไม่ใช่คำสั่งภาษา Python)
#   - pip install : คำสั่งติดตั้งไลบรารีของ Python
#   - -q (quiet)  : ให้แสดงข้อความระหว่างติดตั้งให้น้อยที่สุด
#   - gradio_client   : ไลบรารีสำหรับติดต่อกับเว็บแอปตารางอันดับ (Hugging Face Space)
#   - huggingface_hub : ไลบรารีสำหรับดาวน์โหลดไฟล์จากเว็บไซต์ Hugging Face
!pip -q install gradio_client huggingface_hub

# สร้างตัวแปร (ช่องเก็บค่า) HF_OWNER แล้วใส่ข้อความ "wdittaya" ไว้
#   เครื่องหมาย = หมายถึง "นำค่าทางขวา ไปเก็บไว้ในชื่อทางซ้าย"
#   ข้อความในภาษา Python ต้องอยู่ในเครื่องหมายคำพูด " "
HF_OWNER  = "wdittaya"     # บัญชี Hugging Face ที่ใช้โฮสต์ตารางอันดับของรายวิชา (ผู้สอนเป็นผู้กำหนด)
# รหัสของปฏิบัติการนี้ ใช้ระบุว่าผลที่ส่งเป็นของปฏิบัติการใด (ไม่ต้องแก้ไข)
LAB_ID    = "lab06"
# ชื่อที่จะแสดงบนตารางอันดับ: พิมพ์ชื่อไว้ระหว่างเครื่องหมาย " " เช่น NAME = "หมอเอ"
NAME      = ""            # ไม่บังคับ: ชื่อที่จะแสดงบนตารางอันดับแบบสาธารณะ (สามารถใช้นามแฝงได้)
# รหัสเข้าร่วมตารางอันดับ: พิมพ์รหัสที่ผู้สอนแจ้งไว้ระหว่างเครื่องหมาย " "
JOIN_CODE = ""            # ไม่บังคับ: รหัสเข้าร่วมซึ่งผู้สอนประกาศในการเรียนการสอนผ่าน Zoom

# สร้างชื่อเว็บแอปตารางอันดับ โดยนำ HF_OWNER มาต่อกับข้อความ
#   - f"..." (f-string) คือข้อความที่แทรกค่าตัวแปรได้ โดยเขียนชื่อตัวแปรไว้ใน { }
#   - ผลลัพธ์คือ "wdittaya/aidh-leaderboard"
SPACE_ID  = f"{HF_OWNER}/aidh-leaderboard"
# สร้างชื่อคลังข้อมูล (dataset) ของรายวิชาในลักษณะเดียวกัน ผลลัพธ์คือ "wdittaya/aidh-lab-data"
DATA_REPO = f"{HF_OWNER}/aidh-lab-data"

# นำเข้า (import) ฟังก์ชัน hf_hub_download จากไลบรารี huggingface_hub
#   ฟังก์ชันนี้ใช้ดาวน์โหลดไฟล์จากเว็บไซต์ Hugging Face มาเก็บไว้ในเครื่อง
from huggingface_hub import hf_hub_download

# ----------------------------------------------------------------------------
# สร้างฟังก์ชัน lab_data สำหรับดาวน์โหลดไฟล์ข้อมูลของรายวิชา
#   - def คือคำสั่งสร้างฟังก์ชันใหม่ ตามด้วยชื่อฟังก์ชันและพารามิเตอร์ในวงเล็บ
#   - path : พารามิเตอร์ (ค่าที่ผู้ใช้ส่งเข้ามา) คือชื่อไฟล์ที่ต้องการ เช่น "pima/test_ids.csv"
#   - บรรทัดที่ย่อหน้าเข้าไปด้านล่าง คือคำสั่งที่อยู่ภายในฟังก์ชัน
# ----------------------------------------------------------------------------
def lab_data(path):
    """ดาวน์โหลดไฟล์ข้อมูลทดสอบสาธารณะของรายวิชา 1 ไฟล์ (เช่น 'pima/test_ids.csv') และคืนค่าตำแหน่งไฟล์ในเครื่อง"""
    # เรียก hf_hub_download(...) เพื่อดาวน์โหลดไฟล์ แล้วส่งตำแหน่งไฟล์ในเครื่องกลับไป (return)
    #   - DATA_REPO           : ชื่อคลังข้อมูลบน Hugging Face ที่เก็บไฟล์
    #   - path                : ชื่อไฟล์ที่ต้องการดาวน์โหลดภายในคลังข้อมูลนั้น
    #   - repo_type="dataset" : ระบุว่าคลังนี้เป็นประเภท "ชุดข้อมูล" (ไม่ใช่โมเดล)
    return hf_hub_download(DATA_REPO, path, repo_type="dataset")

# ----------------------------------------------------------------------------
# สร้างฟังก์ชัน submit_to_leaderboard สำหรับส่งไฟล์ผลการทำนายไปยังตารางอันดับ
#   - path   : ชื่อไฟล์ผลการทำนายที่ต้องการส่ง
#   - lab=LAB_ID : รหัสปฏิบัติการ หากไม่ระบุ จะใช้ค่า LAB_ID ที่กำหนดไว้ด้านบนโดยอัตโนมัติ
# ----------------------------------------------------------------------------
def submit_to_leaderboard(path, lab=LAB_ID):
    """ส่งไฟล์ผลการทำนายไปยังตารางอันดับของรายวิชา และแสดงคะแนนพร้อมอันดับ"""
    # นำเข้าเครื่องมือ 2 ตัวจากไลบรารี gradio_client
    #   - Client      : ใช้เชื่อมต่อกับเว็บแอปตารางอันดับ
    #   - handle_file : ใช้เตรียมไฟล์ในเครื่องให้อยู่ในรูปแบบที่ส่งขึ้นเว็บแอปได้
    from gradio_client import Client, handle_file
    # assert คือการตรวจสอบเงื่อนไข หากเงื่อนไขเป็นเท็จ โปรแกรมจะหยุดและแสดงข้อความเตือน
    #   - NAME and JOIN_CODE : เงื่อนไขว่าต้องกรอกทั้ง NAME และ JOIN_CODE แล้ว (ไม่เป็นข้อความว่าง)
    #   - ข้อความหลังเครื่องหมายจุลภาค คือข้อความเตือนที่จะแสดงเมื่อยังไม่ได้กรอก
    assert NAME and JOIN_CODE, "โปรดระบุ NAME และ JOIN_CODE ในเซลล์การตั้งค่าก่อน (จำเป็นเฉพาะการส่งผลไปยังตารางอันดับ)"
    # เชื่อมต่อกับเว็บแอปตารางอันดับ แล้วเก็บการเชื่อมต่อไว้ในตัวแปร client
    #   - SPACE_ID      : ชื่อเว็บแอปตารางอันดับที่กำหนดไว้ด้านบน
    #   - verbose=False : ไม่ต้องแสดงข้อความรายละเอียดระหว่างเชื่อมต่อ
    client = Client(SPACE_ID, verbose=False)
    # ส่งข้อมูลไปยังเว็บแอปด้วย client.predict(...) แล้วแสดงผลลัพธ์ (คะแนนและอันดับ) ด้วย print(...)
    #   - lab=lab                 : รหัสปฏิบัติการ
    #   - name=NAME               : ชื่อที่จะแสดงบนตารางอันดับ
    #   - code=JOIN_CODE          : รหัสเข้าร่วม
    #   - file=handle_file(path)  : ไฟล์ผลการทำนายที่เตรียมพร้อมสำหรับส่ง
    #   - api_name="/submit"      : ชื่อช่องทางรับข้อมูลของเว็บแอป (ส่วน "ส่งผล")
    print(client.predict(lab=lab, name=NAME, code=JOIN_CODE, file=handle_file(path), api_name="/submit"))

# แสดงข้อความยืนยันว่าการตั้งค่าเสร็จแล้ว ด้วยฟังก์ชัน print(...) (พิมพ์ข้อความออกทางหน้าจอ)
#   - ข้อความเป็น f-string จึงแทรกค่า SPACE_ID, LAB_ID และ NAME ลงในข้อความได้
#   - NAME or '(ยังไม่ได้ระบุ ...)' : หาก NAME ว่าง จะแสดงข้อความในวงเล็บแทน
print(f"Leaderboard: https://huggingface.co/spaces/{SPACE_ID}  |  lab={LAB_ID}  name={NAME or '(ยังไม่ได้ระบุ — ตารางอันดับไม่บังคับ)'}")

## ส่วนที่ 1 — การรู้จำเสียงพูดอัตโนมัติด้วย Whisper

การบันทึกเวชระเบียนเป็นภาระงานสำคัญของบุคลากรทางการแพทย์ **การรู้จำเสียงพูดอัตโนมัติ (automatic speech recognition, ASR)** คือระบบที่แปลงเสียงพูดเป็นข้อความ เช่น ระบบผู้ช่วยบันทึกทางการแพทย์ด้วยปัญญาประดิษฐ์ (AI medical scribe) ซึ่งเป็นการประยุกต์ใช้ที่สำคัญด้านสุขภาพดิจิทัล

ปฏิบัติการนี้ใช้ **Whisper** ซึ่งเป็นโมเดลเสียงพูดของ OpenAI ที่เปิดเผยค่าน้ำหนักของโมเดล (**open weights**) ไฟล์โมเดลจะถูกดาวน์โหลดมาประมวลผลภายใน Colab ของผู้เรียนเอง จึงไม่มีการส่งไฟล์เสียงไปยังบริการภายนอก และไม่ต้องใช้ API key Whisper มีหลายขนาดดังนี้

| ขนาด | จำนวนพารามิเตอร์ | ความเร็ว | ความแม่นยำ |
|---|---|---|---|
| `tiny` | 39 M | เร็วที่สุด | ต่ำที่สุด |
| `base` | 74 M | เร็ว | ปานกลาง |
| `small` | 244 M | ปานกลาง | ดีขึ้น |
| `medium` / `turbo` / `large` | 769 M – 1.5 B | ช้าบน Colab รุ่นไม่มีค่าใช้จ่าย | ดีที่สุด |

เซลล์ถัดไปจะติดตั้ง Whisper และโหลดโมเดลขนาด `base` (ใช้เวลาประมาณ 1 นาที)

In [ ]:
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   เซลล์นี้เตรียมเครื่องมือสำหรับ "การรู้จำเสียงพูดอัตโนมัติ" (automatic speech
#   recognition, ASR) ซึ่งเป็นหัวใจของปฏิบัติการนี้ โดยทำ 4 อย่าง ได้แก่
#   1) ติดตั้งไลบรารี openai-whisper (โมเดลถอดเสียงเป็นข้อความ) และ jiwer (ใช้คำนวณ WER)
#   2) นำเข้า (import) ไลบรารีต่าง ๆ ที่ต้องใช้ตลอดปฏิบัติการ
#   3) ตรวจว่าเครื่อง Colab มีการ์ดประมวลผลกราฟิก (GPU) หรือไม่ แล้วโหลดโมเดล Whisper ขนาด "base"
#   4) สร้างฟังก์ชันช่วย 3 ตัว ได้แก่
#        - normalise(...)  : ปรับข้อความให้เป็นรูปแบบมาตรฐานก่อนเปรียบเทียบ
#        - wer_report(...) : คำนวณและแสดงอัตราความผิดพลาดระดับคำ (Word Error Rate, WER)
#        - listen(...)     : แสดงเครื่องเล่นเสียงให้ผู้เรียนกดฟังคลิปได้
#   ผลลัพธ์ที่ผู้เรียนจะเห็น: ข้อความว่าพบ GPU หรือไม่ และข้อความยืนยันว่าโหลดโมเดลเรียบร้อยแล้ว
#   (ครั้งแรกอาจใช้เวลาประมาณ 1-2 นาที เพราะต้องดาวน์โหลดโมเดลขนาดประมาณ 140 MB)
# ============================================================================

# ----------------------------------------------------------------------------
# ส่วนที่ 1: ติดตั้งและนำเข้าไลบรารี
# ----------------------------------------------------------------------------
# ติดตั้งไลบรารีเพิ่มเติมลงในเครื่อง Colab
#   - เครื่องหมาย ! ด้านหน้า หมายถึงสั่งงานระบบปฏิบัติการ (ไม่ใช่คำสั่งภาษา Python)
#   - pip install    : คำสั่งติดตั้งไลบรารีของ Python
#   - -q (quiet)     : ให้แสดงข้อความระหว่างติดตั้งให้น้อยที่สุด
#   - openai-whisper : โมเดล Whisper ของบริษัท OpenAI สำหรับถอดเสียงพูดเป็นข้อความ (รันในเครื่องได้ ไม่ต้องใช้ API key)
#   - jiwer          : ไลบรารีสำหรับคำนวณ WER โดยนับคำที่ผิด คำที่ขาด และคำที่เกิน
!pip -q install openai-whisper jiwer
# นำเข้า (import) ไลบรารีมาตรฐานของ Python 3 ตัวในบรรทัดเดียว (คั่นด้วยเครื่องหมายจุลภาค ,)
#   - re   : เครื่องมือค้นหา/แทนที่ข้อความตามรูปแบบ (regular expression)
#   - time : เครื่องมือเกี่ยวกับเวลา ใช้จับเวลาว่าโมเดลทำงานนานเท่าใด
#   - json : เครื่องมือแปลงข้อมูลให้อยู่ในรูปแบบ JSON (รูปแบบข้อความที่มีโครงสร้าง อ่านง่าย)
import re, time, json
# นำเข้าไลบรารี pandas สำหรับจัดการข้อมูลแบบตาราง
#   - import ... as pd : ตั้งชื่อย่อว่า pd เพื่อพิมพ์สั้นลง เช่น pd.read_csv(...)
import pandas as pd
# นำเข้าไลบรารีอีก 3 ตัว
#   - torch  : ไลบรารี PyTorch สำหรับการเรียนรู้เชิงลึก (deep learning) ซึ่ง Whisper ใช้ทำงานเบื้องหลัง
#   - whisper: ไลบรารี Whisper ที่เพิ่งติดตั้ง ใช้โหลดโมเดลและถอดเสียง
#   - jiwer  : ไลบรารีคำนวณ WER ที่เพิ่งติดตั้ง
import torch, whisper, jiwer
# นำเข้าเครื่องมือ 2 ตัวจากไลบรารี IPython (ระบบเบื้องหลังของโน้ตบุ๊ก Jupyter/Colab)
#   - Audio   : สร้างเครื่องเล่นเสียงที่กดฟังได้ในโน้ตบุ๊ก
#   - display : สั่งแสดงผลวัตถุ (เช่น เครื่องเล่นเสียง หรือตาราง) ออกทางหน้าจอ
from IPython.display import Audio, display

# ----------------------------------------------------------------------------
# ส่วนที่ 2: ตรวจหา GPU และโหลดโมเดล Whisper
# ----------------------------------------------------------------------------
# ตรวจว่าเครื่องนี้มี GPU (การ์ดประมวลผลกราฟิกที่คำนวณได้เร็ว) ให้ใช้งานหรือไม่ แล้วเก็บผลไว้ในตัวแปร USE_FP16
#   - torch.cuda.is_available() : ฟังก์ชันของ PyTorch คืนค่า True (จริง) หากมี GPU และ False (เท็จ) หากไม่มี
#   - ชื่อ FP16 หมายถึงการคำนวณแบบ half precision (ตัวเลขทศนิยม 16 บิต) ซึ่งเร็วกว่าแต่ใช้ได้เฉพาะบน GPU
#   - ข้อความหลังเครื่องหมาย # ท้ายบรรทัดคือหมายเหตุ (comment) ที่ Python ไม่นำไปทำงาน
USE_FP16 = torch.cuda.is_available()      # การคำนวณแบบ half precision ใช้ได้เฉพาะบน GPU
# แสดงผลว่าพบ GPU หรือไม่ ด้วย print(...) ซึ่งพิมพ์ค่าหลายค่าต่อกันโดยเว้นวรรคให้อัตโนมัติ
#   - "พบ GPU:" : ข้อความนำหน้า
#   - USE_FP16  : ค่า True หรือ False
#   - "" if USE_FP16 else "(...)" : นิพจน์เงื่อนไขแบบบรรทัดเดียว หากมี GPU จะแสดงข้อความว่าง
#                                   หากไม่มี จะแสดงคำเตือนว่าใช้ CPU ซึ่งทำงานช้ากว่า
print("พบ GPU:", USE_FP16, "" if USE_FP16 else "(ใช้ CPU: ทำงานได้ แต่ช้ากว่า)")
# โหลดโมเดล Whisper มาเก็บไว้ในตัวแปร model (ครั้งแรกจะดาวน์โหลดค่าน้ำหนักของโมเดลจากอินเทอร์เน็ต)
#   - whisper.load_model(...) : ฟังก์ชันของไลบรารี whisper สำหรับโหลดโมเดล
#   - "base" : ขนาดของโมเดล (มีตั้งแต่ tiny, base, small, medium, large) ขนาดใหญ่ยิ่งแม่นยำแต่ยิ่งช้า
model = whisper.load_model("base")        # ขนาดเหมาะสมกับ Colab รุ่นไม่มีค่าใช้จ่าย

# --- ฟังก์ชันช่วยที่ใช้ตลอดปฏิบัติการ ------------------------------------------------
# ----------------------------------------------------------------------------
# ส่วนที่ 3: ฟังก์ชัน normalise — ปรับข้อความให้เป็นรูปแบบมาตรฐาน
#   - def คือคำสั่งสร้างฟังก์ชันใหม่ (ชุดคำสั่งที่เรียกใช้ซ้ำได้)
#   - text : พารามิเตอร์ คือข้อความที่ต้องการปรับ
#   เหตุผล: เพื่อไม่ให้ความต่างของตัวพิมพ์ใหญ่-เล็กหรือเครื่องหมายวรรคตอน ถูกนับเป็นคำผิด
# ----------------------------------------------------------------------------
def normalise(text):
    """แปลงเป็นตัวพิมพ์เล็กและตัดเครื่องหมายวรรคตอน - วิธีเดียวกับที่ตารางอันดับใช้ก่อนให้คะแนน"""
    # ปรับข้อความทีละขั้นตามลำดับ (เรียกฟังก์ชันต่อกันเป็นสาย) แล้วเก็บผลกลับไปที่ตัวแปร text
    #   1) str(text)          : แปลงค่าที่รับมาให้เป็นข้อความ (เผื่อกรณีได้รับค่าที่ไม่ใช่ข้อความ)
    #   2) .lower()           : เปลี่ยนตัวอักษรภาษาอังกฤษทั้งหมดเป็นตัวพิมพ์เล็ก
    #   3) .replace("-", " ") : แทนเครื่องหมายขีด - ด้วยช่องว่าง (เช่น "follow-up" -> "follow up")
    #   4) .replace("/", " ") : แทนเครื่องหมาย / ด้วยช่องว่าง
    text = str(text).lower().replace("-", " ").replace("/", " ")
    # ลบเครื่องหมายวรรคตอนทั้งหมดด้วย re.sub(รูปแบบ, ข้อความแทนที่, ข้อความต้นฉบับ) จากไลบรารี re
    #   - r"[^\w\s]" : รูปแบบที่หมายถึง "อักขระใด ๆ ที่ไม่ใช่ตัวอักษร/ตัวเลข (\w) และไม่ใช่ช่องว่าง (\s)"
    #                  เช่น . , ? ! ตัว r ด้านหน้าบอกให้ Python อ่านเครื่องหมาย \ ตามตัวอักษร
    #   - ""         : แทนที่ด้วยข้อความว่าง (คือลบทิ้ง)
    #   - text       : ข้อความที่จะนำมาปรับ
    text = re.sub(r"[^\w\s]", "", text)
    # ยุบช่องว่างที่ซ้อนกันหลายช่องให้เหลือช่องเดียว แล้วตัดช่องว่างหัว-ท้ายทิ้ง และส่งผลกลับ (return)
    #   - r"\s+"  : รูปแบบ "ช่องว่างตั้งแต่ 1 ตัวขึ้นไปติดกัน"
    #   - " "     : แทนที่ด้วยช่องว่าง 1 ช่อง
    #   - .strip(): ตัดช่องว่างที่หัวและท้ายข้อความ
    return re.sub(r"\s+", " ", text).strip()

# ----------------------------------------------------------------------------
# ส่วนที่ 4: ฟังก์ชัน wer_report — คำนวณและแสดงค่า WER
#   - reference  : ข้อความอ้างอิงที่ถูกต้อง (มนุษย์ถอดไว้)
#   - hypothesis : ข้อความที่ระบบถอดได้
#   - label=""   : ข้อความนำหน้าผลลัพธ์ (ไม่บังคับ หากไม่ระบุจะเป็นข้อความว่าง)
#   WER = (คำที่ถูกแทนผิด + คำที่ขาด + คำที่เกิน) / จำนวนคำในข้อความอ้างอิง  ค่ายิ่งต่ำยิ่งดี
# ----------------------------------------------------------------------------
def wer_report(reference, hypothesis, label=""):
    """แสดงค่า WER ของข้อความที่ถอดได้ (hypothesis) เทียบกับข้อความที่ถูกต้อง (reference)"""
    # ปรับข้อความทั้งสองด้วย normalise(...) แล้วเก็บไว้ในตัวแปร ref และ hyp พร้อมกันในบรรทัดเดียว
    #   (ค่าทางขวาสองค่าที่คั่นด้วย , จะถูกใส่ลงในตัวแปรทางซ้ายตามลำดับ)
    ref, hyp = normalise(reference), normalise(hypothesis)
    # เปรียบเทียบข้อความทีละคำด้วย jiwer.process_words(...) แล้วเก็บผลการเปรียบเทียบไว้ใน out
    #   - ref : ข้อความอ้างอิง
    #   - hyp if hyp else "<empty>" : หาก hyp ว่างเปล่า (โมเดลถอดไม่ได้เลย) ให้ใช้คำ "<empty>" แทน
    #                                 เพื่อป้องกันข้อผิดพลาดเมื่อส่งข้อความว่างเข้าไป
    #   ผลลัพธ์ out มีค่า out.wer (อัตราความผิดพลาด), out.substitutions (คำที่ถูกแทนผิด),
    #   out.deletions (คำที่ขาด) และ out.insertions (คำที่เกิน)
    out = jiwer.process_words(ref, hyp if hyp else "<empty>")
    # แสดงผลด้วย f-string 2 ส่วนที่เขียนต่อกัน (ข้อความในวงเล็บที่อยู่ติดกันจะถูกรวมเป็นข้อความเดียว)
    #   - {label}           : ข้อความนำหน้า
    #   - {out.wer:.1%}     : แสดง WER เป็นเปอร์เซ็นต์ ทศนิยม 1 ตำแหน่ง (เช่น 0.125 -> 12.5%)
    #   - {out.substitutions}, {out.deletions}, {out.insertions} : จำนวนคำที่ผิด/ขาด/เกิน
    #   - {len(ref.split())} : จำนวนคำในข้อความอ้างอิง (.split() แยกข้อความเป็นรายการคำ, len() นับจำนวน)
    print(f"{label}WER = {out.wer:.1%}  (คำที่ผิด {out.substitutions}, คำที่ขาด {out.deletions}, "
          f"คำที่เกิน {out.insertions} คำ; ข้อความอ้างอิงมี {len(ref.split())} คำ)")
    # ส่งค่า WER (ตัวเลขระหว่าง 0 ขึ้นไป เช่น 0.125) กลับไปให้ผู้เรียกใช้ฟังก์ชัน
    return out.wer

# ----------------------------------------------------------------------------
# ส่วนที่ 5: ฟังก์ชัน listen — แสดงเครื่องเล่นเสียง
#   - path : ตำแหน่ง (ชื่อ) ไฟล์เสียงในเครื่อง
# ----------------------------------------------------------------------------
def listen(path):
    """แสดงเครื่องเล่นเสียงสำหรับคลิป"""
    # สร้างเครื่องเล่นเสียงจากไฟล์ด้วย Audio(filename=path) แล้วแสดงในโน้ตบุ๊กด้วย display(...)
    #   - filename=path : ระบุไฟล์เสียงที่ต้องการเล่น
    display(Audio(filename=path))

# แสดงข้อความยืนยันว่าโหลดโมเดลและสร้างฟังก์ชันเรียบร้อยแล้ว
print("โหลดโมเดล Whisper 'base' เรียบร้อยแล้ว")

💡 **คำอธิบายผลลัพธ์** Colab ดาวน์โหลดโปรแกรม Whisper และค่าน้ำหนักของโมเดล `base` (ประมาณ 140 MB) มาไว้ใน session ของผู้เรียน และได้สร้างฟังก์ชันช่วย 3 ฟังก์ชัน ได้แก่ `listen()` สำหรับแสดงเครื่องเล่นเสียง `normalise()` สำหรับแปลงตัวพิมพ์และตัดเครื่องหมายวรรคตอน และ `wer_report()` สำหรับให้คะแนนข้อความที่ถอดได้ (อธิบายในหัวข้อถัดไป)

In [ ]:
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   ทดลองใช้ Whisper ถอดความเสียงพูดเป็นครั้งแรก กับคลิปตัวอย่างสาธารณะที่บันทึกเสียงได้ชัดเจน
#   (สุนทรพจน์ของประธานาธิบดี J.F. Kennedy) เพื่อให้ผู้เรียนเห็นว่า ASR ทำงานอย่างไร
#   - ข้อมูลเข้า : ไฟล์เสียง sample.flac ที่ดาวน์โหลดจากอินเทอร์เน็ต
#   - ผลลัพธ์   : เครื่องเล่นเสียงให้กดฟัง และข้อความภาษาอังกฤษที่ Whisper ถอดได้
# ============================================================================

# คลิปตัวอย่างสาธารณะ (สุนทรพจน์รับตำแหน่งของ J.F. Kennedy ซึ่งรวมอยู่ในไฟล์ทดสอบของ Whisper)
# ดาวน์โหลดไฟล์เสียงด้วยคำสั่ง wget ของระบบปฏิบัติการ (เครื่องหมาย ! = สั่งงานระบบ ไม่ใช่ Python)
#   - -q             : ไม่ต้องแสดงความคืบหน้าระหว่างดาวน์โหลด
#   - -O sample.flac : บันทึกไฟล์ที่ได้ในชื่อ sample.flac (.flac คือรูปแบบไฟล์เสียงที่ไม่ลดคุณภาพ)
#   - https://...    : ที่อยู่ของไฟล์เสียงบนเว็บไซต์ GitHub
!wget -q -O sample.flac https://github.com/openai/whisper/raw/main/tests/jfk.flac
# แสดงเครื่องเล่นเสียงของไฟล์ sample.flac ด้วยฟังก์ชัน listen(...) ที่สร้างไว้ในเซลล์ก่อนหน้า
listen("sample.flac")
# สั่งให้โมเดล Whisper ถอดความไฟล์เสียง ด้วยเมธอด model.transcribe(...) แล้วเก็บผลไว้ในตัวแปร result
#   - "sample.flac"   : ไฟล์เสียงที่ต้องการถอดความ
#   - fp16=USE_FP16   : ใช้การคำนวณแบบ half precision หรือไม่ (True เมื่อมี GPU, False เมื่อใช้ CPU)
#   - language="en"   : บอกโมเดลว่าเสียงเป็นภาษาอังกฤษ (ไม่ต้องเสียเวลาเดาภาษาเอง)
#   ผลลัพธ์ result เป็น dict (ข้อมูลแบบ "ชื่อ: ค่า") ที่มีข้อความทั้งหมดอยู่ใน result["text"]
result = model.transcribe("sample.flac", fp16=USE_FP16, language="en")
# แสดงข้อความที่ถอดได้
#   - result["text"] : การดึงค่าจาก dict โดยระบุชื่อ (key) ในวงเล็บเหลี่ยม ในที่นี้คือข้อความที่ถอดได้
print("ข้อความที่ Whisper ถอดได้:", result["text"])

### การวัดคุณภาพด้วย WER

**อัตราความผิดพลาดระดับคำ (Word Error Rate, WER)** เปรียบเทียบข้อความที่ระบบถอดได้ (*hypothesis*) กับข้อความที่มนุษย์ถอดไว้อย่างถูกต้อง (*reference* หรือข้อความอ้างอิง)

$$\text{WER} = \frac{\text{คำที่ถูกแทนผิด (substitution)} + \text{คำที่ขาด (deletion)} + \text{คำที่เกิน (insertion)}}{\text{จำนวนคำในข้อความอ้างอิง}}$$

ตัวอย่าง: ข้อความอ้างอิง *"no known drug allergies"* (4 คำ) เทียบกับข้อความที่ถอดได้ *"known drug allergy"* มีคำที่ขาด 1 คำ ("no") และคำที่ถูกแทนผิด 1 คำ ("allergies → allergy") รวมผิด 2 คำจาก 4 คำ คิดเป็น **50 %**

**ค่ายิ่งต่ำยิ่งดี** (0 % หมายถึงถูกต้องทั้งหมด) โดยไม่นับความแตกต่างของตัวพิมพ์ใหญ่-เล็กและเครื่องหมายวรรคตอน การนำไปใช้ทางคลินิกโดยทั่วไปต้องการค่า WER ต่ำกว่า 10 % อย่างชัดเจนสำหรับเสียงพูดในบริบททางการแพทย์

In [ ]:
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   วัดคุณภาพการถอดความของ Whisper ในเซลล์ก่อนหน้า ด้วยอัตราความผิดพลาดระดับคำ
#   (Word Error Rate, WER) โดยเปรียบเทียบกับข้อความอ้างอิง (reference) ที่ถูกต้อง
#   - ข้อมูลเข้า : ข้อความอ้างอิงของสุนทรพจน์ และข้อความที่ Whisper ถอดได้ (result["text"])
#   - ผลลัพธ์   : ค่า WER เป็นเปอร์เซ็นต์ พร้อมจำนวนคำที่ผิด ขาด และเกิน (ค่ายิ่งต่ำยิ่งดี)
# ============================================================================

# สร้างตัวแปร reference เก็บข้อความอ้างอิงที่ถูกต้องของสุนทรพจน์
#   - ข้อความ 2 ชิ้นที่อยู่ในวงเล็บ ( ) และวางติดกัน Python จะนำมาต่อกันเป็นข้อความเดียว
#     (เป็นวิธีเขียนข้อความยาวให้แบ่งเป็นหลายบรรทัดได้)
reference = ("And so my fellow Americans ask not what your country can do for you "
             "ask what you can do for your country")
# เรียกฟังก์ชัน wer_report(...) เพื่อคำนวณและแสดงค่า WER
#   - reference      : ข้อความอ้างอิงที่ถูกต้อง
#   - result["text"] : ข้อความที่ Whisper ถอดได้จากเซลล์ก่อนหน้า
#   - เครื่องหมาย ; ท้ายบรรทัด : ปกติ Jupyter จะแสดงค่าของบรรทัดสุดท้ายของเซลล์โดยอัตโนมัติ
#     (ในที่นี้คือค่า WER ที่ฟังก์ชันส่งกลับ) การใส่ ; ช่วยซ่อนค่านั้น เพื่อไม่ให้แสดงซ้ำ
wer_report(reference, result["text"]);

💡 **คำอธิบายผลลัพธ์** สำหรับเสียงพูดที่ชัดเจนและบันทึกด้วยคุณภาพดี Whisper มักถอดความได้ถูกต้องทั้งหมดหรือเกือบทั้งหมด (WER ใกล้ 0 %) แต่เสียงในสถานการณ์ทางคลินิกจริงซึ่งมีสำเนียง เสียงรบกวน และศัพท์ทางการแพทย์นั้นยากกว่ามาก ส่วนถัดไปจึงใช้เสียงของผู้ป่วยจริง

## ส่วนที่ 2 — การฟังคลิปเสียงของผู้ป่วยจริง

รายวิชาจัดเตรียม **คลิปเสียงสั้น 30 คลิป (ความยาว 3–10 วินาที) ของผู้ที่บรรยายอาการของตนเป็นภาษาอังกฤษ** เช่น *"my knee hurts when I climb stairs"* จากชุดข้อมูล (dataset) สาธารณะ *Medical Speech, Transcription, and Intent* ผู้พูดมีสำเนียงและใช้ไมโครโฟนที่หลากหลาย เซลล์ถัดไปจะดาวน์โหลดรายการคลิปและเล่นคลิปแรก

In [ ]:
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   โหลด "รายการคลิป" (manifest) ของคลิปเสียงผู้ป่วยจริง 30 คลิป ที่บรรยายอาการเป็นภาษาอังกฤษ
#   และสร้างฟังก์ชัน clip_path(...) สำหรับดาวน์โหลดคลิปตามรหัส (เช่น "C07")
#   - ข้อมูลเข้า : ไฟล์ asr/manifest.csv จากคลังข้อมูลของรายวิชา
#   - ผลลัพธ์   : จำนวนคลิปและความยาวรวม ตาราง 5 แถวแรกของรายการคลิป และเครื่องเล่นเสียงของคลิป C01
# ============================================================================

# ดาวน์โหลดไฟล์รายการคลิปแล้วอ่านเป็นตาราง (DataFrame) เก็บไว้ในตัวแปร manifest
#   - lab_data("asr/manifest.csv") : ฟังก์ชันจากเซลล์การตั้งค่า ดาวน์โหลดไฟล์และคืนตำแหน่งไฟล์ในเครื่อง
#   - pd.read_csv(...)             : ฟังก์ชันของ pandas อ่านไฟล์ CSV (ข้อมูลคั่นด้วยจุลภาค) เป็นตาราง
#   หมายเหตุท้ายบรรทัดบอกว่าตารางมี 3 คอลัมน์: id (รหัสคลิป), file (ชื่อไฟล์), seconds (ความยาวเป็นวินาที)
manifest = pd.read_csv(lab_data("asr/manifest.csv"))    # คอลัมน์: id, file, seconds

# ----------------------------------------------------------------------------
# สร้างฟังก์ชัน clip_path สำหรับดาวน์โหลดคลิปเสียง 1 คลิป
#   - clip_id : พารามิเตอร์ คือรหัสคลิป เช่น "C07"
# ----------------------------------------------------------------------------
def clip_path(clip_id):
    """ดาวน์โหลดคลิป 1 คลิป (เช่น 'C07') และคืนค่าตำแหน่งไฟล์ในเครื่อง"""
    # สร้างชื่อไฟล์ด้วย f-string (แทรกค่า clip_id ลงใน { }) เช่น "asr/clips/C07.wav"
    # แล้วดาวน์โหลดด้วย lab_data(...) และส่งตำแหน่งไฟล์ในเครื่องกลับไป (return)
    #   - .wav คือรูปแบบไฟล์เสียงมาตรฐานที่ไม่บีบอัด
    return lab_data(f"asr/clips/{clip_id}.wav")

# แสดงจำนวนคลิปและความยาวรวมของทุกคลิป
#   - len(manifest)             : นับจำนวนแถวของตาราง (= จำนวนคลิป)
#   - manifest["seconds"]       : เลือกคอลัมน์ seconds ด้วยการระบุชื่อคอลัมน์ในวงเล็บเหลี่ยม
#   - .sum()                    : รวมค่าทั้งคอลัมน์ (ความยาวรวมเป็นวินาที)
#   - round(...)                : ปัดเศษให้เป็นจำนวนเต็ม
print("จำนวนคลิป", len(manifest), "คลิป ความยาวรวม", round(manifest["seconds"].sum()), "วินาที")
# แสดงตาราง 5 แถวแรกของรายการคลิป
#   - manifest.head() : เมธอดของ pandas คืนค่าแถวแรก ๆ ของตาราง (ค่าเริ่มต้น 5 แถว)
#   - display(...)    : แสดงตารางในรูปแบบที่อ่านง่าย
display(manifest.head())
# ดาวน์โหลดคลิป C01 ด้วย clip_path("C01") แล้วแสดงเครื่องเล่นเสียงด้วย listen(...) ให้ผู้เรียนกดฟัง
listen(clip_path("C01"))

In [ ]:
# ⭐ Exercise 1 — แบบฝึกหัดที่ 1: เลือกคลิปผู้ป่วย ฟัง และให้ Whisper ถอดความ
# หน้าที่ของเซลล์: เล่นคลิปที่เลือก และแสดงข้อความที่โมเดล Whisper 'base' ถอดได้
# สิ่งที่ต้องแก้ไข: แทน ... ด้วยรหัสคลิประหว่าง "C01" ถึง "C30" ภายในเครื่องหมายอัญประกาศ
# คำใบ้:  clip_id = "C17"
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   ให้ผู้เรียนเลือกคลิปเสียงผู้ป่วย 1 คลิป ฟังด้วยตนเอง แล้วให้ Whisper (ขนาด base) ถอดความ
#   เพื่อเปรียบเทียบสิ่งที่หูได้ยินกับสิ่งที่เครื่องถอดได้ทีละคำ
#   - ข้อมูลเข้า : รหัสคลิปที่ผู้เรียนเลือก (clip_id)
#   - ผลลัพธ์   : เครื่องเล่นเสียงของคลิปนั้น และข้อความที่ Whisper ถอดได้
# ============================================================================

# กำหนดรหัสคลิปที่ต้องการฟังไว้ในตัวแปร clip_id
#   - ... (จุดสามจุด) คือช่องว่างที่ผู้เรียนต้องเติมเอง ตามคำแนะนำด้านบน
#   - # TODO เป็นหมายเหตุบอกว่าบรรทัดนี้ "ต้องทำ"
clip_id = ...      # TODO

# ดาวน์โหลดคลิปตามรหัสด้วย clip_path(clip_id) แล้วแสดงเครื่องเล่นเสียงด้วย listen(...)
listen(clip_path(clip_id))
# ให้ Whisper ถอดความคลิป แล้วเก็บข้อความไว้ในตัวแปร clip_text โดยทำต่อกันเป็นขั้น ๆ
#   1) clip_path(clip_id)     : ตำแหน่งไฟล์เสียงของคลิปที่เลือก
#   2) model.transcribe(...)  : ถอดความเสียง
#        - fp16=USE_FP16      : ใช้การคำนวณแบบ half precision เมื่อมี GPU
#        - language="en"      : ระบุว่าเสียงเป็นภาษาอังกฤษ
#   3) ["text"]               : ดึงเฉพาะข้อความที่ถอดได้จากผลลัพธ์ (dict)
#   4) .strip()               : ตัดช่องว่างที่หัวและท้ายข้อความ
clip_text = model.transcribe(clip_path(clip_id), fp16=USE_FP16, language="en")["text"].strip()
# แสดงข้อความที่ถอดได้ โดยใช้ f-string แทรกรหัสคลิป {clip_id} ลงในข้อความนำ
print(f"ข้อความที่ Whisper (base) ถอดได้จาก {clip_id}:", clip_text)

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้ (เฉลยแบบฝึกหัดที่ 1)
#   เลือกคลิป "C17" ซึ่งมีศัพท์ทางการแพทย์ แล้วเล่นเสียงและให้ Whisper (ขนาด base) ถอดความ
#   - ผลลัพธ์ : เครื่องเล่นเสียงของคลิป C17 และข้อความที่ Whisper ถอดได้
#   (บรรทัดหัวข้อ (@ title) ด้านบนทำให้ Colab แสดงหัวข้อเซลล์ และซ่อนโค้ดไว้ ดับเบิลคลิกเพื่อดูโค้ด)
# ============================================================================

# เฉลย: กำหนดรหัสคลิปเป็น "C17" (ข้อความต้องอยู่ในเครื่องหมายอัญประกาศ " ")
clip_id = "C17"

# ดาวน์โหลดคลิปตามรหัสด้วย clip_path(clip_id) แล้วแสดงเครื่องเล่นเสียงด้วย listen(...)
listen(clip_path(clip_id))
# ให้ Whisper ถอดความคลิป แล้วเก็บข้อความไว้ในตัวแปร clip_text
#   1) clip_path(clip_id)     : ตำแหน่งไฟล์เสียงของคลิป
#   2) model.transcribe(...)  : ถอดความเสียง
#        - fp16=USE_FP16      : ใช้การคำนวณแบบ half precision เมื่อมี GPU
#        - language="en"      : ระบุว่าเสียงเป็นภาษาอังกฤษ
#   3) ["text"]               : ดึงเฉพาะข้อความจากผลลัพธ์ (dict)
#   4) .strip()               : ตัดช่องว่างที่หัวและท้ายข้อความ
clip_text = model.transcribe(clip_path(clip_id), fp16=USE_FP16, language="en")["text"].strip()
# แสดงข้อความที่ถอดได้ โดยใช้ f-string แทรกรหัสคลิปลงในข้อความนำ
print(f"ข้อความที่ Whisper (base) ถอดได้จาก {clip_id}:", clip_text)

💡 **คำอธิบายผลลัพธ์** Whisper แปลงเสียงเป็นข้อความภายในเวลาประมาณหนึ่งถึงสองวินาที ให้เล่นคลิปอีกครั้ง (คลิกปุ่ม ▶ บนเครื่องเล่น) และเปรียบเทียบกับข้อความที่แสดงทีละคำ ผู้เรียนสามารถทดลองกับคลิปอื่นได้โดยเปลี่ยนค่า `clip_id` แล้วรันเซลล์ใหม่

✍️ **คำถามที่ 1** โปรดเขียนข้อความที่ผู้เรียนได้ยินจากคลิปให้ตรงที่สุด และพิจารณาว่า Whisper ถอดความได้ถูกต้องทุกคำหรือไม่ หากไม่ถูกต้อง โปรดระบุคำที่ผิด และอธิบายสาเหตุที่เป็นไปได้ (เช่น สำเนียง ความเร็วในการพูด เสียงรบกวน หรือศัพท์ทางการแพทย์ที่พบไม่บ่อย)

✍️ **คำตอบ:**



In [ ]:
# ⭐ Exercise 2 — แบบฝึกหัดที่ 2: เปรียบเทียบ Whisper ขนาดต่าง ๆ บนคลิปชุดเดียวกัน
# หน้าที่ของเซลล์: ถอดความ 6 คลิป (คลิปที่เลือก และอีก 5 คลิปที่มีศัพท์ทางการแพทย์) ด้วยโมเดลแต่ละขนาด พร้อมจับเวลา
# สิ่งที่ต้องแก้ไข: แทน ... ด้วยขนาดโมเดลที่ต้องการเปรียบเทียบ ภายในเครื่องหมายอัญประกาศ คั่นด้วยจุลภาค
# คำใบ้:  sizes_to_compare = ["tiny", "base", "small"]
#        (ไม่ควรใช้ "medium"/"large" บน Colab รุ่นไม่มีค่าใช้จ่าย เนื่องจากทำงานช้าและใช้หน่วยความจำจำนวนมาก)
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   เปรียบเทียบความแม่นยำและความเร็วของโมเดล Whisper หลายขนาด บนคลิปเสียงชุดเดียวกัน
#   เพื่อให้เห็นการแลกเปลี่ยน (trade-off) ระหว่าง "ถอดได้ถูกต้องกว่า" กับ "ทำงานช้ากว่า/ใช้ทรัพยากรมากกว่า"
#   ซึ่งสำคัญเมื่อโรงพยาบาลต้องเลือกระบบที่รันบนเครื่องของตนเอง
#   - ข้อมูลเข้า : รายชื่อขนาดโมเดลที่ผู้เรียนเลือก และคลิป 6 คลิป
#   - ผลลัพธ์   : เวลาเฉลี่ยต่อคลิปของแต่ละโมเดล และตารางข้อความที่แต่ละโมเดลถอดได้ (1 คอลัมน์ต่อ 1 โมเดล)
# ============================================================================

# ----------------------------------------------------------------------------
# ส่วนที่ 1: กำหนดโมเดลและคลิปที่จะเปรียบเทียบ
# ----------------------------------------------------------------------------
# กำหนดรายการ (list) ขนาดโมเดลที่ต้องการเปรียบเทียบ
#   - list คือกลุ่มของค่าหลายค่าเรียงกัน เขียนในวงเล็บเหลี่ยม [ ] และคั่นแต่ละค่าด้วยจุลภาค
#   - [...] : ช่องว่างที่ผู้เรียนต้องเติมชื่อขนาดโมเดลเอง ตามคำใบ้ด้านบน
sizes_to_compare = [...]      # TODO

# สร้างรายการรหัสคลิปที่จะเปรียบเทียบ เก็บไว้ในตัวแปร compare_ids
#   - { ... } ที่มีค่าเรียงกันโดยไม่มีเครื่องหมาย : คือเซต (set) ซึ่งตัดค่าที่ซ้ำกันออกโดยอัตโนมัติ
#     (เช่น หากผู้เรียนเลือก clip_id = "C17" ไว้แล้ว จะไม่มี "C17" ซ้ำสองครั้ง)
#   - clip_id : คลิปที่ผู้เรียนเลือกในแบบฝึกหัดที่ 1
#   - "C05", "C12", ... : คลิปอื่นที่มีศัพท์ทางการแพทย์
#   - sorted(...) : เรียงลำดับค่าจากน้อยไปมาก และคืนค่าเป็น list
compare_ids = sorted({clip_id, "C05", "C12", "C17", "C25", "C28"})
# สร้าง dict (ข้อมูลแบบ "ชื่อ: ค่า" เขียนใน { } ) ชื่อ models สำหรับเก็บโมเดลที่โหลดแล้ว
#   - "base": model : ใส่โมเดล base ที่โหลดไว้แล้วในเซลล์ตั้งต้น เพื่อไม่ต้องโหลดซ้ำ
models = {"base": model}
# สร้างตาราง (DataFrame) ชื่อ table ที่มีคอลัมน์แรกชื่อ "clip" เก็บรหัสคลิป
#   - pd.DataFrame({...}) : สร้างตารางจาก dict โดยชื่อ (key) เป็นชื่อคอลัมน์ และค่าเป็นข้อมูลในคอลัมน์
table = pd.DataFrame({"clip": compare_ids})

# ----------------------------------------------------------------------------
# ส่วนที่ 2: วนซ้ำถอดความด้วยโมเดลทีละขนาด พร้อมจับเวลา
#   - for size in sizes_to_compare: คือการวนทำคำสั่งที่ย่อหน้าเข้าไปด้านล่าง ซ้ำทีละค่า
#     ในรอบแรก size = ค่าแรกของรายการ รอบถัดไปเป็นค่าถัดไป จนครบทุกค่า
# ----------------------------------------------------------------------------
for size in sizes_to_compare:
    # if ... : ตรวจเงื่อนไข หากเป็นจริงจึงทำคำสั่งที่ย่อหน้าเข้าไป
    #   - size not in models : "ยังไม่มีโมเดลขนาดนี้ใน dict models" (ยังไม่เคยโหลด)
    if size not in models:
        # แจ้งผู้เรียนว่ากำลังโหลดโมเดล (f-string แทรกชื่อขนาดโมเดลลงใน { })
        print(f"กำลังโหลดโมเดล '{size}' ...")
        # โหลดโมเดลขนาดนั้นด้วย whisper.load_model(size) แล้วเก็บลงใน dict models โดยใช้ชื่อขนาดเป็น key
        models[size] = whisper.load_model(size)
    # บันทึกเวลาเริ่มต้น ด้วย time.time() (คืนค่าเวลาปัจจุบันเป็นจำนวนวินาที)
    start = time.time()
    # ถอดความทุกคลิปด้วยโมเดลขนาดนี้ แล้วเก็บผลเป็นคอลัมน์ใหม่ในตาราง โดยใช้ชื่อขนาดโมเดลเป็นชื่อคอลัมน์
    #   - [ ... for c in compare_ids ] : list comprehension คือการสร้าง list โดยวนทีละคลิป c
    #     แล้วนำผลลัพธ์ของแต่ละรอบมาเรียงต่อกัน
    #   - models[size]             : เลือกโมเดลตามขนาดจาก dict
    #   - .transcribe(clip_path(c), fp16=USE_FP16, language="en") : ถอดความคลิป c (ภาษาอังกฤษ)
    #   - ["text"].strip()         : ดึงข้อความที่ถอดได้ และตัดช่องว่างหัว-ท้าย
    #   - table[size] = ...        : สร้าง (หรือแทนที่) คอลัมน์ชื่อ size ในตาราง
    table[size] = [models[size].transcribe(clip_path(c), fp16=USE_FP16, language="en")["text"].strip()
                   for c in compare_ids]
    # แสดงเวลาเฉลี่ยต่อคลิปของโมเดลขนาดนี้
    #   - (time.time() - start) : เวลาที่ใช้ทั้งหมด (เวลาปัจจุบัน ลบ เวลาเริ่มต้น)
    #   - / len(compare_ids)    : หารด้วยจำนวนคลิป ได้เวลาเฉลี่ยต่อคลิป
    #   - {size:>6}             : แสดงชื่อขนาดโมเดลชิดขวาในความกว้าง 6 ตัวอักษร (ให้ตัวเลขตรงแนวกัน)
    #   - :.1f                  : แสดงตัวเลขทศนิยม 1 ตำแหน่ง
    print(f"{size:>6}: {(time.time() - start) / len(compare_ids):.1f} วินาทีต่อคลิป")

# ----------------------------------------------------------------------------
# ส่วนที่ 3: แสดงตารางเปรียบเทียบ
# ----------------------------------------------------------------------------
# ตั้งค่าการแสดงผลของ pandas ด้วย pd.set_option(...)
#   - "display.max_colwidth" : ความกว้างสูงสุดของข้อความในแต่ละช่องของตาราง
#   - None                   : ไม่จำกัดความกว้าง จึงแสดงข้อความที่ถอดได้ครบทั้งประโยค (ไม่ถูกตัดเป็น ...)
pd.set_option("display.max_colwidth", None)
# แสดงตารางเปรียบเทียบ: แต่ละแถวคือ 1 คลิป แต่ละคอลัมน์คือข้อความจากโมเดลแต่ละขนาด
display(table)

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้ (เฉลยแบบฝึกหัดที่ 2)
#   เปรียบเทียบโมเดล Whisper 3 ขนาด ได้แก่ tiny, base และ small บนคลิปเสียงชุดเดียวกัน 6 คลิป
#   เพื่อดูการแลกเปลี่ยน (trade-off) ระหว่างความแม่นยำกับความเร็ว
#   - ผลลัพธ์ : เวลาเฉลี่ยต่อคลิปของแต่ละโมเดล และตารางข้อความที่แต่ละโมเดลถอดได้
#   (ไม่เลือก medium/large เพราะช้าและใช้หน่วยความจำมากเกินไปสำหรับ Colab รุ่นไม่มีค่าใช้จ่าย)
# ============================================================================

# ----------------------------------------------------------------------------
# ส่วนที่ 1: กำหนดโมเดลและคลิปที่จะเปรียบเทียบ
# ----------------------------------------------------------------------------
# เฉลย: รายการ (list) ขนาดโมเดลที่จะเปรียบเทียบ เรียงจากเล็กไปใหญ่ (ข้อความแต่ละค่าอยู่ใน " " คั่นด้วย ,)
sizes_to_compare = ["tiny", "base", "small"]

# สร้างรายการรหัสคลิปที่จะเปรียบเทียบ
#   - { ... } : เซต (set) ซึ่งตัดค่าที่ซ้ำกันออกโดยอัตโนมัติ (clip_id = "C17" จึงไม่ซ้ำ)
#   - sorted(...) : เรียงลำดับและคืนค่าเป็น list
compare_ids = sorted({clip_id, "C05", "C12", "C17", "C25", "C28"})
# สร้าง dict ชื่อ models เก็บโมเดลที่โหลดแล้ว เริ่มจากโมเดล base ที่โหลดไว้ในเซลล์ตั้งต้น
models = {"base": model}
# สร้างตาราง (DataFrame) ที่มีคอลัมน์ "clip" เก็บรหัสคลิป
table = pd.DataFrame({"clip": compare_ids})

# ----------------------------------------------------------------------------
# ส่วนที่ 2: วนซ้ำ (for) ถอดความด้วยโมเดลทีละขนาด พร้อมจับเวลา
# ----------------------------------------------------------------------------
for size in sizes_to_compare:
    # หากยังไม่เคยโหลดโมเดลขนาดนี้ (ไม่มีอยู่ใน dict models) ให้โหลดก่อน
    if size not in models:
        # แจ้งผู้เรียนว่ากำลังโหลดโมเดลขนาดใด
        print(f"กำลังโหลดโมเดล '{size}' ...")
        # โหลดโมเดลด้วย whisper.load_model(size) แล้วเก็บลงใน dict models
        models[size] = whisper.load_model(size)
    # บันทึกเวลาเริ่มต้น (วินาที) ด้วย time.time()
    start = time.time()
    # ถอดความทุกคลิปด้วย list comprehension ([ ... for c in compare_ids ] = วนทีละคลิปแล้วรวมผลเป็น list)
    # แล้วเก็บเป็นคอลัมน์ใหม่ชื่อตามขนาดโมเดล
    #   - .transcribe(clip_path(c), fp16=USE_FP16, language="en") : ถอดความคลิป c เป็นภาษาอังกฤษ
    #   - ["text"].strip() : ดึงข้อความและตัดช่องว่างหัว-ท้าย
    table[size] = [models[size].transcribe(clip_path(c), fp16=USE_FP16, language="en")["text"].strip()
                   for c in compare_ids]
    # แสดงเวลาเฉลี่ยต่อคลิป = (เวลาปัจจุบัน - เวลาเริ่มต้น) / จำนวนคลิป
    #   - {size:>6} : ชื่อโมเดลชิดขวาในความกว้าง 6 ตัวอักษร
    #   - :.1f      : ทศนิยม 1 ตำแหน่ง
    print(f"{size:>6}: {(time.time() - start) / len(compare_ids):.1f} วินาทีต่อคลิป")

# ----------------------------------------------------------------------------
# ส่วนที่ 3: แสดงตารางเปรียบเทียบ
# ----------------------------------------------------------------------------
# ตั้งค่า pandas ให้แสดงข้อความในแต่ละช่องเต็มความยาว
#   - "display.max_colwidth" : ความกว้างสูงสุดของข้อความในแต่ละช่อง
#   - None                   : ไม่จำกัด (ไม่ตัดข้อความ)
pd.set_option("display.max_colwidth", None)
# แสดงตารางเปรียบเทียบ (แถว = คลิป, คอลัมน์ = ขนาดโมเดล)
display(table)

💡 **คำอธิบายผลลัพธ์** แต่ละคอลัมน์คือข้อความที่โมเดลแต่ละขนาดถอดจากคลิปเดียวกัน โมเดลที่มีขนาดใหญ่กว่าได้เรียนรู้รูปแบบของเสียงพูดมากกว่า จึงมักรองรับสำเนียงและศัพท์ทางการแพทย์ (*phlegm*, *anemia*, *cruciate ligament*) ได้ดีกว่า แต่ทำงานช้ากว่าและใช้หน่วยความจำมากกว่า ให้ใช้เครื่องเล่นเสียงในเซลล์ถัดไปเพื่อตรวจสอบว่าโมเดลใดถอดความได้ถูกต้อง

In [ ]:
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   แสดงเครื่องเล่นเสียงของทุกคลิปที่ใช้เปรียบเทียบในเซลล์ก่อนหน้า เพื่อให้ผู้เรียนฟังเสียงจริง
#   แล้วตัดสินด้วยตนเองว่าโมเดลขนาดใดถอดความได้ใกล้เคียงที่สุด (ผู้เรียนทำหน้าที่เป็นมาตรฐานอ้างอิง)
#   - ผลลัพธ์ : รหัสคลิปและเครื่องเล่นเสียง เรียงกันทีละคลิป
# ============================================================================

# เครื่องเล่นเสียงของคลิปที่ใช้เปรียบเทียบ - ให้ฟังและพิจารณาว่าคอลัมน์ใดใกล้เคียงกับเสียงพูดมากที่สุด
# วนซ้ำ (for) ทีละรหัสคลิป c ในรายการ compare_ids
for c in compare_ids:
    # ในบรรทัดนี้มี 2 คำสั่ง คั่นด้วยเครื่องหมาย ; (อัฒภาค)
    #   1) print(c)             : แสดงรหัสคลิป
    #   2) listen(clip_path(c)) : แสดงเครื่องเล่นเสียงของคลิปนั้น
    print(c); listen(clip_path(c))

In [ ]:
# @title 🎧 ไม่บังคับ: ผู้เรียนเป็นผู้ถอดความ — พิมพ์ข้อความที่ได้ยินจากคลิปที่เลือก (เป็นภาษาอังกฤษ) แล้วรันเซลล์นี้
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้ (ไม่บังคับ)
#   ให้ผู้เรียนเป็นผู้ถอดความคลิปที่เลือก (clip_id) ด้วยตนเอง แล้วใช้ข้อความนั้นเป็น
#   "ข้อความอ้างอิง" (reference / gold standard) เพื่อคำนวณ WER ของโมเดลแต่ละขนาด
#   - ข้อมูลเข้า : ข้อความที่ผู้เรียนพิมพ์ในช่อง my_reference และตาราง table จากแบบฝึกหัดที่ 2
#   - ผลลัพธ์   : ค่า WER ของแต่ละโมเดล (หรือข้อความเตือน หากยังไม่ได้พิมพ์ข้อความ)
# ============================================================================

# สร้างตัวแปร my_reference เก็บข้อความที่ผู้เรียนถอดเอง (เริ่มต้นเป็นข้อความว่าง "")
#   - หมายเหตุแบบฟอร์ม (@ param ชนิด string) ท้ายบรรทัดถัดไป ทำให้ Colab แสดง "ช่องกรอกข้อความ" ทางขวาของเซลล์
#     ผู้เรียนพิมพ์ข้อความในช่องนั้นได้เลย โดยไม่ต้องแก้โค้ด (ห้ามแก้หรือลบหมายเหตุนี้)
my_reference = ""  # @param {type:"string"}

# ตรวจว่าผู้เรียนพิมพ์ข้อความแล้วหรือยัง
#   - my_reference.strip() : ตัดช่องว่างหัว-ท้าย หากเหลือเป็นข้อความว่าง จะถือว่าเป็น "เท็จ"
#   - not ...              : กลับค่าความจริง จึงหมายถึง "ถ้ายังไม่ได้พิมพ์ข้อความ"
if not my_reference.strip():
    # แสดงคำแนะนำให้ผู้เรียนพิมพ์ข้อความที่ได้ยินจากคลิป แล้วรันเซลล์ใหม่ (f-string แทรกรหัสคลิป)
    print(f"โปรดพิมพ์ข้อความที่ได้ยินจาก {clip_id} ในช่อง my_reference (หรือภายในเครื่องหมายอัญประกาศ) แล้วรันเซลล์ใหม่ เพื่อแสดงค่า WER ของแต่ละโมเดล")
# else: กรณีที่ผู้เรียนพิมพ์ข้อความแล้ว ให้ทำคำสั่งที่ย่อหน้าเข้าไปด้านล่าง
else:
    # วนซ้ำทีละขนาดโมเดลที่เปรียบเทียบไว้ในแบบฝึกหัดที่ 2
    for size in sizes_to_compare:
        # ดึงข้อความที่โมเดลขนาดนี้ถอดได้จากคลิปที่เลือก ออกมาจากตาราง table เก็บไว้ใน hyp
        #   - table["clip"] == clip_id : สร้างเงื่อนไข True/False ทีละแถว ว่าแถวนั้นเป็นคลิปที่เลือกหรือไม่
        #   - table.loc[เงื่อนไขแถว, size] : เลือกแถวที่ตรงเงื่อนไข และคอลัมน์ชื่อ size
        #   - .iloc[0] : เลือกค่าแรก (ตำแหน่งที่ 0 เพราะ Python เริ่มนับจาก 0) ให้ได้ข้อความเดียว
        hyp = table.loc[table["clip"] == clip_id, size].iloc[0]
        # คำนวณและแสดง WER ของโมเดลนี้ โดยใช้ข้อความของผู้เรียนเป็นข้อความอ้างอิง
        #   - my_reference : ข้อความอ้างอิงที่ผู้เรียนถอดเอง
        #   - hyp          : ข้อความที่โมเดลถอดได้
        #   - label=f"{size:>6}: " : ข้อความนำหน้าเป็นชื่อโมเดล ชิดขวาในความกว้าง 6 ตัวอักษร
        wer_report(my_reference, hyp, label=f"{size:>6}: ")

💡 **คำอธิบายผลลัพธ์** การคำนวณ WER จำเป็นต้องมีข้อความที่มนุษย์ถอดไว้อย่างถูกต้อง ในกรณีนี้ผู้เรียนทำหน้าที่เป็นมาตรฐานอ้างอิง (gold standard) ในโครงการจริง แพทย์ พยาบาล หรือผู้ถอดความที่ผ่านการฝึกอบรมจะจัดทำข้อความอ้างอิงสำหรับชุดทดสอบก่อนที่เครื่องมือ medical scribe จะได้รับการอนุมัติให้ใช้งาน

✍️ **คำถามที่ 2** จากผลลัพธ์ข้างต้น โมเดลขนาดใดถอดความคลิปชุดนี้ได้ดีที่สุด และทำงานช้ากว่า `tiny` เพียงใด หากโรงพยาบาลต้องการใช้ระบบรู้จำเสียงพูดบนคอมพิวเตอร์ทั่วไปที่ไม่มี GPU (เพื่อไม่ให้ไฟล์เสียงของผู้ป่วยออกนอกโรงพยาบาล) ผู้เรียนจะเลือกโมเดลขนาดใด และต้องยอมรับข้อจำกัดใดบ้าง จงอธิบาย

✍️ **คำตอบ:**



## ส่วนที่ 3 — ข้อจำกัดของ WER: ความผิดพลาดที่เป็นอันตรายทางคลินิก

WER นับคำที่ผิดทุกคำด้วยน้ำหนักเท่ากัน การขาดคำว่า *"the"* และการขาดคำว่า *"no"* ต่างนับเป็นความผิดพลาด 1 คำ แต่มีเพียงกรณีหลังที่อาจเป็นอันตรายต่อผู้ป่วย ความผิดพลาดของ ASR ที่เป็นอันตรายและพบบ่อย ได้แก่ **การปฏิเสธ (negation)** (*"no allergies"* → *"allergies"*) **ตัวเลขและหน่วย** (*"15"* ↔ *"50"*, *"mg"* ↔ *"mcg"*) **ชื่อยาที่ออกเสียงคล้ายกัน (sound-alike drugs)** (*hydroxyzine* ↔ *hydralazine*) และ **ข้างซ้าย/ขวา**

In [ ]:
# ⭐ Exercise 3 — แบบฝึกหัดที่ 3: สร้างความผิดพลาด "เพียงเล็กน้อย" ที่เป็นอันตรายทางคลินิก
# หน้าที่ของเซลล์: เปรียบเทียบข้อความถอดความที่ผิดพลาด 2 แบบของคำบอกบันทึก (dictation) เดียวกัน
#                 ด้วยค่า WER และด้วยการตรวจสอบคำสำคัญทางคลินิก
# สิ่งที่ต้องแก้ไข: แทน ... ด้วยสำเนาของ dictation ที่เปลี่ยนหรือลบคำเพียง 1 คำ
#                 จนข้อความมีความหมายที่เป็นอันตรายต่อผู้ป่วย
# คำใบ้:  dangerous_hyp = "Patient has known drug allergies. Start metoprolol 25 milligrams twice daily."
#        (ตัวอย่างอื่น: 25 -> 250, twice -> three times)
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   แสดงข้อจำกัดของ WER: WER นับทุกคำที่ผิดด้วยน้ำหนักเท่ากัน แต่ความผิดพลาดบางคำ
#   (เช่น การปฏิเสธ (negation) คำว่า "no" ตัวเลขขนาดยา หรือหน่วย) อาจเป็นอันตรายต่อผู้ป่วยมาก
#   เซลล์นี้จึงเปรียบเทียบข้อความถอดความ 2 แบบ ได้แก่ แบบผิดแต่ไม่เป็นอันตราย (harmless)
#   และแบบที่ผู้เรียนสร้างให้เป็นอันตราย (dangerous) ทั้งด้วยค่า WER และด้วยการตรวจ "คำสำคัญทางคลินิก"
#   - ผลลัพธ์ : ค่า WER ของแต่ละแบบ และรายการคำสำคัญที่ขาดหายไป
# ============================================================================

# ----------------------------------------------------------------------------
# ส่วนที่ 1: กำหนดข้อความต้นฉบับและข้อความถอดความ 2 แบบ
# ----------------------------------------------------------------------------
# ข้อความที่แพทย์บอกบันทึกจริง (dictation) ใช้เป็นข้อความอ้างอิงที่ถูกต้อง
#   ความหมาย: ผู้ป่วยไม่มีประวัติแพ้ยา เริ่มยา metoprolol 25 มิลลิกรัม วันละ 2 ครั้ง
dictation     = "Patient has no known drug allergies. Start metoprolol 25 milligrams twice daily."
# ข้อความถอดความที่ผิดหลายคำ (มี "The" เกิน และ "daily" กลายเป็น "a day") แต่ความหมายทางคลินิกยังถูกต้อง
harmless_hyp  = "The patient has no known drug allergies, start metoprolol 25 milligrams twice a day."
# ข้อความถอดความที่เป็นอันตราย: ... คือช่องว่างที่ผู้เรียนต้องเติมเอง
#   โดยคัดลอก dictation แล้วเปลี่ยนหรือลบคำเพียง 1 คำ ตามคำแนะนำด้านบน (ใส่ในเครื่องหมาย " ")
dangerous_hyp = ...      # TODO

# รายการ (list) คำสำคัญทางคลินิกที่ต้องถอดได้ถูกต้อง (คำปฏิเสธ ชื่อยา ขนาดยา หน่วย และความถี่)
critical_words = ["no", "metoprolol", "25", "milligrams", "twice"]   # คำสำคัญที่บุคลากรทางการแพทย์ต้องได้รับอย่างถูกต้อง

# ----------------------------------------------------------------------------
# ส่วนที่ 2: ฟังก์ชัน missing_critical — หาคำสำคัญที่ขาดหายไปจากข้อความถอดความ
#   - hypothesis : ข้อความที่ระบบถอดได้
#   คืนค่า: list ของคำสำคัญที่ไม่พบในข้อความ
# ----------------------------------------------------------------------------
def missing_critical(hypothesis):
    # ปรับข้อความด้วย normalise(...) (ตัวพิมพ์เล็ก ไม่มีวรรคตอน) แล้วแยกเป็นรายการคำด้วย .split()
    hyp_words = normalise(hypothesis).split()
    # คืนค่า list ของคำสำคัญที่ไม่พบ ด้วย list comprehension
    #   - for w in critical_words : วนทีละคำสำคัญ w
    #   - if w not in hyp_words   : เก็บไว้เฉพาะคำที่ "ไม่อยู่" ในรายการคำของข้อความถอดความ
    return [w for w in critical_words if w not in hyp_words]

# ----------------------------------------------------------------------------
# ส่วนที่ 3: เปรียบเทียบข้อความทั้ง 2 แบบ
#   - [("harmless ", harmless_hyp), ("dangerous", dangerous_hyp)] : list ของคู่ค่า (tuple) 2 คู่
#     แต่ละคู่คือ (ป้ายชื่อ, ข้อความถอดความ)
#   - for label, hyp in ... : วนทีละคู่ และแยกค่าในคู่ใส่ตัวแปร label และ hyp
# ----------------------------------------------------------------------------
for label, hyp in [("harmless ", harmless_hyp), ("dangerous", dangerous_hyp)]:
    # คำนวณและแสดง WER เทียบกับ dictation โดยมีป้ายชื่อนำหน้า (label=f"{label}: ")
    wer_report(dictation, hyp, label=f"{label}: ")
    # แสดงคำสำคัญที่ขาดหายไป
    #   - missing_critical(hyp) or "ไม่มี" : หาก list ว่าง (ไม่มีคำสำคัญขาด) จะแสดงคำว่า "ไม่มี" แทน
    print("           คำสำคัญที่ขาดหายไป:", missing_critical(hyp) or "ไม่มี")

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้ (เฉลยแบบฝึกหัดที่ 3)
#   สร้างข้อความถอดความที่เป็นอันตราย โดย "ลบคำว่า no เพียงคำเดียว" ทำให้
#   "ไม่มีประวัติแพ้ยา" กลายเป็น "มีประวัติแพ้ยา" แล้วเปรียบเทียบกับข้อความที่ผิดแต่ไม่เป็นอันตราย
#   ผลที่ได้แสดงว่าข้อความที่เป็นอันตรายมีค่า WER "ต่ำกว่า" (ดูเหมือนดีกว่า) แต่ขาดคำสำคัญ "no"
#   จึงต้องตรวจคำสำคัญทางคลินิกควบคู่กับ WER เสมอ
#   - ผลลัพธ์ : ค่า WER ของแต่ละแบบ และรายการคำสำคัญที่ขาดหายไป
# ============================================================================

# ----------------------------------------------------------------------------
# ส่วนที่ 1: กำหนดข้อความต้นฉบับและข้อความถอดความ 2 แบบ
# ----------------------------------------------------------------------------
# ข้อความที่แพทย์บอกบันทึกจริง (dictation): ไม่มีประวัติแพ้ยา เริ่ม metoprolol 25 มิลลิกรัม วันละ 2 ครั้ง
dictation     = "Patient has no known drug allergies. Start metoprolol 25 milligrams twice daily."
# ข้อความที่ผิดหลายคำ ("The" เกิน, "daily" -> "a day") แต่ความหมายทางคลินิกยังถูกต้อง
harmless_hyp  = "The patient has no known drug allergies, start metoprolol 25 milligrams twice a day."
# เฉลย: คัดลอก dictation แล้วลบคำว่า "no" ออก 1 คำ ความหมายกลับเป็น "ผู้ป่วยมีประวัติแพ้ยา"
dangerous_hyp = "Patient has known drug allergies. Start metoprolol 25 milligrams twice daily."

# รายการ (list) คำสำคัญทางคลินิกที่ต้องถอดได้ถูกต้อง
critical_words = ["no", "metoprolol", "25", "milligrams", "twice"]   # คำสำคัญที่บุคลากรทางการแพทย์ต้องได้รับอย่างถูกต้อง

# ----------------------------------------------------------------------------
# ส่วนที่ 2: ฟังก์ชัน missing_critical — หาคำสำคัญที่ขาดหายไป
#   - hypothesis : ข้อความที่ระบบถอดได้
# ----------------------------------------------------------------------------
def missing_critical(hypothesis):
    # ปรับข้อความด้วย normalise(...) แล้วแยกเป็นรายการคำด้วย .split()
    hyp_words = normalise(hypothesis).split()
    # คืนค่า list ของคำสำคัญ w ที่ไม่พบในรายการคำ (list comprehension พร้อมเงื่อนไข if)
    return [w for w in critical_words if w not in hyp_words]

# ----------------------------------------------------------------------------
# ส่วนที่ 3: วนเปรียบเทียบข้อความทั้ง 2 แบบ (แต่ละรอบได้คู่ ป้ายชื่อ label และข้อความ hyp)
# ----------------------------------------------------------------------------
for label, hyp in [("harmless ", harmless_hyp), ("dangerous", dangerous_hyp)]:
    # คำนวณและแสดง WER เทียบกับ dictation พร้อมป้ายชื่อนำหน้า
    wer_report(dictation, hyp, label=f"{label}: ")
    # แสดงคำสำคัญที่ขาดหายไป หาก list ว่างจะแสดง "ไม่มี" แทน (ตัวดำเนินการ or)
    print("           คำสำคัญที่ขาดหายไป:", missing_critical(hyp) or "ไม่มี")

💡 **คำอธิบายผลลัพธ์** ข้อความที่ไม่เป็นอันตราย (harmless) มีคำผิด *มากกว่า* (มีคำว่า "the" เกิน และ "daily" → "a day") จึงมีค่า WER *สูงกว่า* ข้อความที่เป็นอันตราย (dangerous) ทั้งที่ข้อความที่เป็นอันตรายกลับความหมายของประวัติการแพ้ยาเป็นตรงกันข้าม ค่า WER เพียงค่าเดียวจึงไม่สามารถสะท้อนความเสี่ยงนี้ได้ การประเมินทางคลินิกจำเป็นต้องตรวจสอบคำสำคัญด้วย ได้แก่ การปฏิเสธ ชื่อยา ขนาดยา หน่วย และข้างซ้าย/ขวา

✍️ **คำถามที่ 3** จากข้อความถอดความของคลิปผู้ป่วยในส่วนที่ 2 มีโมเดลใดถอดความผิดจนความหมายทางคลินิกเปลี่ยนไปหรือไม่ (เช่น ระบุอวัยวะผิด ระบุข้างผิด หรือคำว่า "not" ขาดหายไป) จากนั้นจงอธิบายใน 2–3 ประโยคว่า ความเสี่ยงที่สำคัญที่สุดของความผิดพลาดของ ASR ในการบันทึกเวชระเบียนคืออะไร และเพราะเหตุใดค่า WER เฉลี่ยเพียงอย่างเดียวจึงไม่เพียงพอสำหรับการอนุมัติเครื่องมือ medical scribe

✍️ **คำตอบ:**



## ส่วนที่ 4 — การแปลงบันทึกเสียง (voice note) เป็นสรุปแบบมีโครงสร้าง

เครื่องมือ medical scribe จะมีประโยชน์มากขึ้นเมื่อสามารถแปลงคำพูดอิสระเป็น **ข้อมูลแบบมีโครงสร้าง (structured fields)** เช่น รายการยา สัญญาณชีพ และประวัติการแพ้ยา ซึ่งบันทึกลงเวชระเบียนได้ ส่วนนี้ใช้ **กฎ (rules)** อย่างง่าย โดย *นิพจน์ปกติ (regular expression, regex)* คือรูปแบบของข้อความ เช่น รูปแบบ "คำหนึ่งคำ ตามด้วยตัวเลข และหน่วย" จะค้นพบข้อความ *"aspirin 81 milligrams"*

บันทึกเสียงด้านล่าง (ภาษาอังกฤษ) เป็นตัวอย่างข้อความที่ Whisper ถอดจากการบอกบันทึกของพยาบาล

In [ ]:
# ⭐ Exercise 4 — แบบฝึกหัดที่ 4: สกัดชื่อยาและขนาดยาจากบันทึกเสียง
# หน้าที่ของเซลล์: ค้นหารูปแบบ "ชื่อยา ตัวเลข หน่วย" (รวมถึงความดันโลหิต อาการที่ผู้ป่วยปฏิเสธ และประวัติการแพ้ยา)
#                 และแสดงสรุปแบบมีโครงสร้าง
# สิ่งที่ต้องแก้ไข: แทน ... ด้วยหน่วยของขนาดยาที่ต้องการค้นหา ภายในเครื่องหมายอัญประกาศ คั่นด้วย |  (หมายถึง "หรือ")
#                 โดยให้คำที่สะกดยาวกว่าอยู่ก่อน
# คำใบ้:  DOSE_UNITS = "milligrams|milligram|mg|micrograms|mcg|units"
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   แปลงบันทึกเสียง (voice note) ที่ถอดเป็นข้อความแล้ว ให้เป็นข้อมูลแบบมีโครงสร้าง (structured fields)
#   ที่บันทึกลงเวชระเบียนได้ ได้แก่ รายการยาและขนาดยา ความดันโลหิต อาการที่ผู้ป่วยปฏิเสธ และประวัติการแพ้ยา
#   โดยใช้ "กฎ" ที่เขียนเป็นนิพจน์ปกติ (regular expression, regex) คือรูปแบบของข้อความที่ต้องการค้นหา
#   - ข้อมูลเข้า : ข้อความ voice_note และรายการหน่วยขนาดยา DOSE_UNITS ที่ผู้เรียนกำหนด
#   - ผลลัพธ์   : สรุปข้อมูลแบบมีโครงสร้างในรูปแบบ JSON
# ============================================================================

# ----------------------------------------------------------------------------
# ส่วนที่ 1: ข้อมูลเข้า
# ----------------------------------------------------------------------------
# ข้อความบันทึกเสียงของพยาบาลที่ Whisper ถอดได้ (ข้อความ 3 ชิ้นในวงเล็บ ( ) ถูกต่อกันเป็นข้อความเดียว)
#   ความหมาย: ผู้ป่วยหญิงอายุ 67 ปี เวียนศีรษะ 2 วัน ไม่เจ็บหน้าอก ความดันโลหิต 150/95
#   ให้ amlodipine 5 mg วันละครั้งต่อ และเริ่ม aspirin 81 มิลลิกรัมทุกวัน แพ้ penicillin
voice_note = ("Patient is a 67 year old woman with dizziness for two days. No chest pain. "
              "Blood pressure 150 over 95. Continue amlodipine 5 mg once daily and start "
              "aspirin 81 milligrams daily. Allergic to penicillin.")

# กำหนดรายการหน่วยของขนาดยาที่จะค้นหา เป็นข้อความเดียวที่คั่นแต่ละหน่วยด้วย | (หมายถึง "หรือ" ใน regex)
#   - ... คือช่องว่างที่ผู้เรียนต้องเติมเอง ตามคำแนะนำด้านบน
DOSE_UNITS = ...      # TODO

# ----------------------------------------------------------------------------
# ส่วนที่ 2: ฟังก์ชัน structure_note — แปลงข้อความอิสระเป็นข้อมูลแบบมีโครงสร้าง
#   - text : ข้อความบันทึกที่ต้องการสกัดข้อมูล
#   คืนค่า: dict ที่มี 4 หัวข้อ ได้แก่ medications, blood_pressure, denied_symptoms, allergies
# ----------------------------------------------------------------------------
def structure_note(text):
    # ค้นหาทุกตำแหน่งที่มีรูปแบบ "ชื่อยา ตัวเลข หน่วย" ด้วย re.findall(รูปแบบ, ข้อความ, flags=...)
    #   ส่วนประกอบของรูปแบบ (นำข้อความ 3 ชิ้นมาต่อกันด้วย + โดยแทรก DOSE_UNITS ไว้ตรงกลาง):
    #   - \b             : ขอบของคำ (ให้เริ่มที่ต้นคำพอดี)
    #   - ([a-z]+)       : กลุ่มที่ 1 = ตัวอักษรตั้งแต่ 1 ตัวขึ้นไป (ชื่อยา)
    #   - \s+            : ช่องว่างอย่างน้อย 1 ช่อง
    #   - (\d+(?:\.\d+)?) : กลุ่มที่ 2 = ตัวเลข อาจมีทศนิยมหรือไม่ก็ได้ (เช่น 5 หรือ 2.5)
    #                      (?: ... )? หมายถึงส่วนที่มีหรือไม่มีก็ได้ และไม่นับเป็นกลุ่มแยก
    #   - \s*            : ช่องว่าง 0 ช่องขึ้นไป (เช่น "5mg" หรือ "5 mg")
    #   - ( DOSE_UNITS ) : กลุ่มที่ 3 = หน่วยใดหน่วยหนึ่งในรายการ DOSE_UNITS
    #   - \b             : ขอบของคำ (หน่วยต้องจบคำพอดี)
    #   - text           : ข้อความที่จะค้นหา
    #   - flags=re.I     : ไม่สนใจตัวพิมพ์ใหญ่-เล็ก (re.I = IGNORECASE)
    #   ผลลัพธ์ drug_dose เป็น list ของชุดค่า 3 ค่า (ชื่อยา, ตัวเลข, หน่วย)
    drug_dose = re.findall(r"\b([a-z]+)\s+(\d+(?:\.\d+)?)\s*(" + DOSE_UNITS + r")\b", text, flags=re.I)
    # ค้นหาความดันโลหิตตำแหน่งแรกที่พบ ด้วย re.search(...) (คืนผลการค้นพบ หรือ None หากไม่พบ)
    #   - (\d{2,3})   : กลุ่มที่ 1 = ตัวเลข 2-3 หลัก (ความดันตัวบน systolic)
    #   - \s*(?:over|/)\s* : คำว่า "over" หรือเครื่องหมาย / โดยมีช่องว่างหรือไม่ก็ได้
    #   - (\d{2,3})   : กลุ่มที่ 2 = ตัวเลข 2-3 หลัก (ความดันตัวล่าง diastolic)
    #   - flags=re.I  : ไม่สนใจตัวพิมพ์ใหญ่-เล็ก
    bp = re.search(r"(\d{2,3})\s*(?:over|/)\s*(\d{2,3})", text, flags=re.I)
    # คืนค่าเป็น dict (ข้อมูลแบบ "ชื่อ: ค่า") ที่มี 4 หัวข้อ (เขียนแยกบรรทัดเพื่อให้อ่านง่าย)
    return {
        # "medications": รายการยา สร้างด้วย list comprehension วนทีละชุด (d=ชื่อยา, n=ตัวเลข, u=หน่วย)
        #   แต่ละรายการเป็น dict {"drug": ชื่อยาตัวพิมพ์เล็ก, "dose": "ตัวเลข หน่วย"} (.lower() = แปลงเป็นตัวพิมพ์เล็ก)
        "medications": [{"drug": d.lower(), "dose": f"{n} {u.lower()}"} for d, n, u in drug_dose],
        # "blood_pressure": หากพบความดัน (bp ไม่ใช่ None) ให้รวมเป็นข้อความ "ตัวบน/ตัวล่าง"
        #   - bp.group(1), bp.group(2) : ค่าในกลุ่มที่ 1 และ 2 ของผลการค้นพบ
        #   - ... if bp else None      : หากไม่พบ ให้ใส่ None (ไม่มีค่า)
        "blood_pressure": f"{bp.group(1)}/{bp.group(2)}" if bp else None,
        # "denied_symptoms": อาการที่ผู้ป่วยปฏิเสธ ค้นหาทุกตำแหน่งด้วย re.findall(...)
        #   - \bno         : คำว่า "no" ที่ขึ้นต้นคำ
        #   - ([a-z ]+?)   : กลุ่มที่เก็บ = ตัวอักษรหรือช่องว่าง โดยเก็บให้สั้นที่สุด (+? = แบบไม่ตะกละ)
        #   - (?=[.,]|$)   : ต้องตามด้วยจุด จุลภาค หรือท้ายข้อความ (ตรวจดูแต่ไม่นับรวมในผล)
        #   - flags=re.I   : ไม่สนใจตัวพิมพ์ใหญ่-เล็ก
        "denied_symptoms": re.findall(r"\bno ([a-z ]+?)(?=[.,]|$)", text, flags=re.I),
        # "allergies": ยาที่แพ้ = คำที่ตามหลัง "allergic to" (([a-z]+) = ตัวอักษร 1 ตัวขึ้นไป)
        "allergies": re.findall(r"allergic to ([a-z]+)", text, flags=re.I),
    }

# ----------------------------------------------------------------------------
# ส่วนที่ 3: เรียกใช้ฟังก์ชันและแสดงผล
# ----------------------------------------------------------------------------
# เรียก structure_note(voice_note) เพื่อสกัดข้อมูลจากบันทึก แล้วเก็บผลไว้ใน summary
summary = structure_note(voice_note)
# แปลง dict เป็นข้อความรูปแบบ JSON ด้วย json.dumps(...) แล้วแสดงผล
#   - summary  : ข้อมูลที่จะแปลง
#   - indent=2 : จัดย่อหน้าบรรทัดละ 2 ช่องว่าง ให้อ่านง่าย
print(json.dumps(summary, indent=2))

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้ (เฉลยแบบฝึกหัดที่ 4)
#   สกัดข้อมูลแบบมีโครงสร้างจากบันทึกเสียง (รายการยาและขนาดยา ความดันโลหิต อาการที่ผู้ป่วยปฏิเสธ
#   และประวัติการแพ้ยา) ด้วยนิพจน์ปกติ (regular expression, regex)
#   เฉลยกำหนดหน่วยขนาดยา 6 แบบ ได้แก่ milligrams, milligram, mg, micrograms, mcg, units
#   - ผลลัพธ์ : สรุปแบบ JSON ที่พบยา amlodipine 5 mg และ aspirin 81 milligrams,
#               ความดัน 150/95, อาการที่ปฏิเสธ "chest pain" และแพ้ penicillin
# ============================================================================

# ----------------------------------------------------------------------------
# ส่วนที่ 1: ข้อมูลเข้า
# ----------------------------------------------------------------------------
# ข้อความบันทึกเสียงของพยาบาลที่ Whisper ถอดได้ (ข้อความ 3 ชิ้นในวงเล็บถูกต่อกันเป็นข้อความเดียว)
voice_note = ("Patient is a 67 year old woman with dizziness for two days. No chest pain. "
              "Blood pressure 150 over 95. Continue amlodipine 5 mg once daily and start "
              "aspirin 81 milligrams daily. Allergic to penicillin.")

# เฉลย: หน่วยของขนาดยาที่จะค้นหา คั่นด้วย | (หมายถึง "หรือ") โดยเรียงคำที่ยาวกว่าไว้ก่อน
#   (เช่น "milligrams" ก่อน "milligram") เพื่อให้ regex ลองจับคำเต็มก่อน
DOSE_UNITS = "milligrams|milligram|mg|micrograms|mcg|units"

# ----------------------------------------------------------------------------
# ส่วนที่ 2: ฟังก์ชัน structure_note — แปลงข้อความอิสระเป็นข้อมูลแบบมีโครงสร้าง
#   - text : ข้อความบันทึกที่ต้องการสกัดข้อมูล
# ----------------------------------------------------------------------------
def structure_note(text):
    # ค้นหาทุกรูปแบบ "ชื่อยา ตัวเลข หน่วย" ด้วย re.findall(...) ได้ list ของชุด (ชื่อยา, ตัวเลข, หน่วย)
    #   - \b([a-z]+)       : กลุ่มที่ 1 ชื่อยา (ตัวอักษร 1 ตัวขึ้นไป เริ่มที่ต้นคำ)
    #   - \s+              : ช่องว่างอย่างน้อย 1 ช่อง
    #   - (\d+(?:\.\d+)?)  : กลุ่มที่ 2 ตัวเลข อาจมีทศนิยม
    #   - \s*              : ช่องว่าง 0 ช่องขึ้นไป
    #   - (" + DOSE_UNITS + r") : กลุ่มที่ 3 หน่วยใดหน่วยหนึ่งใน DOSE_UNITS
    #   - \b               : หน่วยต้องจบคำพอดี
    #   - flags=re.I       : ไม่สนใจตัวพิมพ์ใหญ่-เล็ก
    drug_dose = re.findall(r"\b([a-z]+)\s+(\d+(?:\.\d+)?)\s*(" + DOSE_UNITS + r")\b", text, flags=re.I)
    # ค้นหาความดันโลหิตตำแหน่งแรกด้วย re.search(...) (คืน None หากไม่พบ)
    #   - (\d{2,3}) ตัวบน, ตามด้วย "over" หรือ "/", แล้ว (\d{2,3}) ตัวล่าง
    #   - flags=re.I : ไม่สนใจตัวพิมพ์ใหญ่-เล็ก
    bp = re.search(r"(\d{2,3})\s*(?:over|/)\s*(\d{2,3})", text, flags=re.I)
    # คืนค่าเป็น dict ที่มี 4 หัวข้อ
    return {
        # รายการยา: วนทีละชุด (d=ชื่อยา, n=ตัวเลข, u=หน่วย) สร้าง dict {"drug": ..., "dose": "n u"} ตัวพิมพ์เล็ก
        "medications": [{"drug": d.lower(), "dose": f"{n} {u.lower()}"} for d, n, u in drug_dose],
        # ความดันโลหิต: "ตัวบน/ตัวล่าง" จากกลุ่มที่ 1 และ 2 หากพบ มิฉะนั้นเป็น None
        "blood_pressure": f"{bp.group(1)}/{bp.group(2)}" if bp else None,
        # อาการที่ปฏิเสธ: ข้อความหลังคำว่า "no" จนถึงจุด จุลภาค หรือท้ายข้อความ (เก็บให้สั้นที่สุดด้วย +?)
        "denied_symptoms": re.findall(r"\bno ([a-z ]+?)(?=[.,]|$)", text, flags=re.I),
        # ประวัติแพ้ยา: คำที่ตามหลัง "allergic to"
        "allergies": re.findall(r"allergic to ([a-z]+)", text, flags=re.I),
    }

# ----------------------------------------------------------------------------
# ส่วนที่ 3: เรียกใช้ฟังก์ชันและแสดงผล
# ----------------------------------------------------------------------------
# สกัดข้อมูลจาก voice_note แล้วเก็บผลไว้ใน summary
summary = structure_note(voice_note)
# แสดงผลในรูปแบบ JSON ด้วย json.dumps(...) (indent=2 = ย่อหน้า 2 ช่องว่างให้อ่านง่าย)
print(json.dumps(summary, indent=2))

💡 **คำอธิบายผลลัพธ์** กฎที่กำหนดค้นพบยาทั้งสองรายการพร้อมขนาดยา ความดันโลหิต อาการที่ผู้ป่วยปฏิเสธ ("no chest pain") และประวัติการแพ้ยา กฎลักษณะนี้โปร่งใสและคาดการณ์ผลได้ แต่ไม่ยืดหยุ่น หากขนาดยาถูกพูดเป็นคำ เช่น *"five milligrams"* หรือชื่อยาถูกถอดความผิด ระบบจะค้นไม่พบหรือบันทึกข้อมูลที่ผิดลงไปโดยตรง เซลล์ถัดไปทดลองใช้กฎชุดเดิมกับข้อความที่มีความผิดพลาดของ ASR

In [ ]:
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   จำลองสถานการณ์ที่ Whisper ถอดความผิดเพียง 2 จุด แล้วส่งข้อความนั้นเข้าสู่กฎสกัดข้อมูลชุดเดิม
#   เพื่อให้เห็นว่าความผิดพลาดของ ASR ส่งผ่านเข้าสู่เวชระเบียนแบบมีโครงสร้างได้โดยไม่มีการแจ้งเตือน
#   (ขนาดยาเพิ่มเป็น 10 เท่า และการปฏิเสธ "no chest pain" หายไป)
#   - ผลลัพธ์ : สรุปแบบ JSON ของข้อความที่ถอดผิด ให้เปรียบเทียบกับผลในเซลล์ก่อนหน้า
# ============================================================================

# บันทึกเดิมที่ Whisper ถอดความผิด: "No chest pain" -> "Now chest pain", "5 mg" -> "50 mg"
# สร้างข้อความที่ถอดผิด โดยแทนที่ข้อความใน voice_note 2 ครั้งต่อกันด้วย .replace(เดิม, ใหม่)
#   1) .replace("No chest pain", "Now chest pain") : เปลี่ยน "No" เป็น "Now" (การปฏิเสธหายไป)
#   2) .replace("5 mg", "50 mg")                    : เปลี่ยนขนาดยา 5 mg เป็น 50 mg
misheard = voice_note.replace("No chest pain", "Now chest pain").replace("5 mg", "50 mg")
# สกัดข้อมูลจากข้อความที่ถอดผิดด้วย structure_note(...) แล้วแสดงเป็น JSON (indent=2 = ย่อหน้า 2 ช่อง)
print(json.dumps(structure_note(misheard), indent=2))

💡 **คำอธิบายผลลัพธ์** pipeline สร้างบันทึกแบบมีโครงสร้างได้ตามปกติ ทั้งที่ **ขนาดยาเพิ่มขึ้นเป็นสิบเท่า** และการปฏิเสธ "no chest pain" หายไปโดยไม่มีการแจ้งเตือน ความผิดพลาดของการรู้จำเสียงพูดจึงส่งผ่านเข้าสู่เวชระเบียนแบบมีโครงสร้างโดยตรง ด้วยเหตุนี้บุคลากรทางการแพทย์จึงต้องตรวจทานและลงนามรับรองบันทึกทุกฉบับที่ปัญญาประดิษฐ์ร่างขึ้น

In [ ]:
# @title 🎙️ ส่วนเสริม (ไม่บังคับ): บอกบันทึกทางคลินิกด้วยเสียงของผู้เรียน (เลือกช่องทำเครื่องหมาย รันเซลล์ แล้วอัปโหลดไฟล์เสียง)
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้ (ไม่บังคับ)
#   ให้ผู้เรียนทดลองอัปโหลดไฟล์เสียงที่บันทึกเอง (ประโยคทางคลินิกสมมติ ภาษาอังกฤษ)
#   แล้วให้ Whisper ถอดความ และสกัดข้อมูลแบบมีโครงสร้างด้วย structure_note(...) จากแบบฝึกหัดที่ 4
#   - ข้อมูลเข้า : ไฟล์เสียงของผู้เรียน (wav/mp3/m4a) — ใช้ข้อมูลสมมติเท่านั้น
#   - ผลลัพธ์   : เครื่องเล่นเสียง ข้อความที่ถอดได้ และสรุปแบบ JSON
#                 (หากไม่ได้เลือกช่องทำเครื่องหมาย เซลล์จะแสดงข้อความว่าข้ามส่วนนี้)
# ============================================================================

# ตัวแปร use_my_recording เป็นค่าจริง/เท็จ (True/False) ว่าต้องการทดลองกับเสียงของตนเองหรือไม่
#   - หมายเหตุแบบฟอร์ม (@ param ชนิด boolean) ท้ายบรรทัดถัดไป ทำให้ Colab แสดง "ช่องทำเครื่องหมาย" (checkbox)
#     ทางขวาของเซลล์ ติ๊กเพื่อเปลี่ยนเป็น True โดยไม่ต้องแก้โค้ด (ห้ามแก้หรือลบหมายเหตุนี้)
use_my_recording = False  # @param {type:"boolean"}
# บันทึกเสียงประโยคสั้น ๆ ภาษาอังกฤษด้วยโทรศัพท์หรือคอมพิวเตอร์ (wav/mp3/m4a) เช่น
#   "Patient presents with acute chest pain and shortness of breath, prescribe aspirin 81 milligrams."
# ใช้ข้อมูลสมมติเท่านั้น - ห้ามอัปโหลดเสียงของผู้ป่วยจริง

# ตรวจเงื่อนไข: หากติ๊กช่อง use_my_recording (เป็น True) ให้ทำคำสั่งที่ย่อหน้าเข้าไปด้านล่าง
if use_my_recording:
    # นำเข้าเครื่องมือ files จากไลบรารี google.colab (ใช้ได้เฉพาะใน Colab) สำหรับอัปโหลด/ดาวน์โหลดไฟล์
    from google.colab import files
    # แสดงปุ่มให้ผู้เรียนเลือกไฟล์จากเครื่องของตน แล้วอัปโหลดขึ้น Colab ด้วย files.upload()
    #   ผลลัพธ์ uploaded เป็น dict ที่มีชื่อไฟล์เป็น key และเนื้อหาไฟล์เป็นค่า (ไฟล์ถูกบันทึกไว้ใน Colab ด้วย)
    uploaded = files.upload()                 # เลือกไฟล์เสียง
    # ดึงชื่อไฟล์แรกที่อัปโหลด เก็บไว้ใน audio_path
    #   - iter(uploaded) : สร้างตัววนอ่านชื่อไฟล์ (key) ทีละชื่อ
    #   - next(...)      : หยิบชื่อแรกออกมา
    audio_path = next(iter(uploaded))
    # แสดงเครื่องเล่นเสียงของไฟล์ที่อัปโหลด
    listen(audio_path)
    # ให้ Whisper ถอดความไฟล์เสียง (fp16=USE_FP16 ใช้ half precision เมื่อมี GPU, language="en" ภาษาอังกฤษ)
    # แล้วดึงเฉพาะข้อความ ["text"] เก็บไว้ใน my_text
    my_text = model.transcribe(audio_path, fp16=USE_FP16, language="en")["text"]
    # แสดงข้อความที่ถอดได้
    print("ข้อความที่ Whisper ถอดได้:", my_text)
    # สกัดข้อมูลแบบมีโครงสร้างด้วย structure_note(...) แล้วแสดงเป็น JSON (indent=2 = ย่อหน้า 2 ช่อง)
    print(json.dumps(structure_note(my_text), indent=2))
# else: กรณีที่ไม่ได้ติ๊กช่อง use_my_recording
else:
    # แสดงข้อความแจ้งว่าข้ามส่วนนี้
    print("ข้ามส่วนนี้ (เลือก use_my_recording หากต้องการทดลองกับเสียงของผู้เรียนเอง)")

✍️ **คำถามที่ 4** สมมติว่าโรงพยาบาลต้องการนำ AI medical scribe ที่ฟังการตรวจรักษาและกรอกข้อมูลรายการยาและประวัติการแพ้ยาโดยอัตโนมัติมาใช้ โปรดระบุ **มาตรการป้องกันความเสี่ยง (safeguards) อย่างน้อย 2 ข้อ** ที่ควรกำหนดทั้งก่อนและหลังเริ่มใช้งานจริง พร้อมเหตุผล (ประเด็นที่ควรพิจารณา ได้แก่ การตรวจทานโดยบุคลากร การทดสอบกับสำเนียงและชื่อยาที่ใช้ในพื้นที่ ความยินยอมของผู้ป่วย และสถานที่ประมวลผลและจัดเก็บไฟล์เสียง)

✍️ **คำตอบ:**



## 🏆 โจทย์ท้าทายบนตารางอันดับ (Leaderboard) — ไม่บังคับ สำหรับผู้เรียนที่เขียนโปรแกรมได้

**ส่วนนี้ไม่บังคับ และไม่นับเป็นชิ้นงานที่ต้องส่ง** ส่วนนี้ถอดความ **คลิปผู้ป่วยทั้ง 30 คลิป** และส่งผลเป็นไฟล์ `lab06_submission.csv` ซึ่งมีคอลัมน์ `id,transcript` **ตัวชี้วัด: ค่า WER รวมของทุกคลิป** (ค่ายิ่งต่ำยิ่งดี โดยไม่นับตัวพิมพ์ใหญ่-เล็กและเครื่องหมายวรรคตอน) ข้อความอ้างอิงที่ถูกต้องถูกเก็บไว้บนเซิร์ฟเวอร์ของตารางอันดับ

1. รันเซลล์ baseline ด้านล่าง ซึ่งจะสร้างไฟล์ส่งผลที่มีรูปแบบถูกต้องทันที (ประมาณ 1 นาทีบน GPU หรือหลายนาทีบน CPU)
2. หากต้องการส่งผล ให้ระบุ `NAME` และ `JOIN_CODE` ในเซลล์การตั้งค่า (และรันเซลล์นั้นใหม่) จากนั้นรันเซลล์ส่งผล
3. ทดลองปรับปรุงให้ได้ผลดีกว่า baseline ในเซลล์ "ระบบที่ปรับปรุงแล้ว"

> **หลักความซื่อสัตย์ทางวิชาการ (Honour code)** ข้อความอ้างอิงที่ซ่อนไว้มาจากชุดข้อมูลสาธารณะ การค้นหาข้อความดังกล่าวจึงทำได้ง่ายและไม่เกิดประโยชน์ต่อการเรียนรู้ ตารางอันดับมีไว้เพื่อสนับสนุนการเรียนรู้ และโน้ตบุ๊ก (notebook) ของผู้เรียนจะได้รับการประเมินด้วย **ห้ามแก้ไขหรือพิมพ์ข้อความถอดความด้วยตนเอง** เนื่องจากวัตถุประสงค์คือการวัดความสามารถของ *ระบบ*

In [ ]:
# 🏆 Baseline — โมเดล Whisper 'base' ภาษาอังกฤษ ค่าตั้งต้น -> lab06_submission.csv
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้ (ส่วนท้าทาย ไม่บังคับ)
#   สร้าง "ผลพื้นฐาน" (baseline) สำหรับตารางอันดับ โดยใช้โมเดล Whisper 'base' ถอดความ
#   คลิปผู้ป่วยทั้ง 30 คลิป แล้วบันทึกเป็นไฟล์ส่งผล lab06_submission.csv (คอลัมน์ id, transcript)
#   - ข้อมูลเข้า : รายการคลิป manifest
#   - ผลลัพธ์   : ไฟล์ lab06_submission.csv และตาราง 5 แถวแรกของผลการถอดความ
#   (ใช้เวลาประมาณ 1 นาทีบน GPU หรือหลายนาทีบน CPU)
# ============================================================================

# สร้าง list ว่าง [] ชื่อ rows สำหรับสะสมผลการถอดความทีละคลิป
rows = []
# วนซ้ำทีละคลิป โดยจับคู่รหัสคลิปกับชื่อไฟล์
#   - zip(manifest["id"], manifest["file"]) : จับคู่ค่าจาก 2 คอลัมน์ทีละแถว เป็นคู่ (รหัส, ชื่อไฟล์)
#   - for cid, f in ... : แยกคู่ใส่ตัวแปร cid (รหัสคลิป) และ f (ชื่อไฟล์)
for cid, f in zip(manifest["id"], manifest["file"]):
    # ดาวน์โหลดคลิปด้วย lab_data(f"asr/{f}") แล้วให้ Whisper ถอดความ เก็บผลไว้ใน out
    #   - fp16=USE_FP16 : ใช้ half precision เมื่อมี GPU
    #   - language="en" : ระบุว่าเป็นภาษาอังกฤษ
    out = model.transcribe(lab_data(f"asr/{f}"), fp16=USE_FP16, language="en")
    # เพิ่มผลของคลิปนี้ต่อท้าย list rows ด้วย .append(...) ในรูป dict {"id": รหัสคลิป, "transcript": ข้อความ}
    #   - out["text"].strip() : ข้อความที่ถอดได้ ตัดช่องว่างหัว-ท้าย
    rows.append({"id": cid, "transcript": out["text"].strip()})
# แปลง list ของ dict เป็นตาราง (DataFrame) ชื่อ sub (1 dict = 1 แถว)
sub = pd.DataFrame(rows)
# บันทึกตารางเป็นไฟล์ CSV ด้วย .to_csv(...)
#   - "lab06_submission.csv" : ชื่อไฟล์ที่จะบันทึก
#   - index=False            : ไม่ต้องบันทึกเลขลำดับแถว (index) ลงในไฟล์
sub.to_csv("lab06_submission.csv", index=False)
# แสดงจำนวนคลิปที่ถอดความแล้ว (len(sub) = จำนวนแถว)
print("ถอดความแล้ว", len(sub), "คลิป -> lab06_submission.csv")
# แสดงตาราง 5 แถวแรกของผลการถอดความ
display(sub.head())

In [ ]:
# 🏆 ส่งผล (เฉพาะกรณีที่ระบุ NAME และ JOIN_CODE ในเซลล์การตั้งค่าแล้ว)
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้ (ส่วนท้าทาย ไม่บังคับ)
#   ส่งไฟล์ lab06_submission.csv ไปยังตารางอันดับของรายวิชา เพื่อรับคะแนน (ค่า WER รวม) และอันดับ
#   - ทำงานเฉพาะเมื่อผู้เรียนกรอก NAME และ JOIN_CODE ในเซลล์การตั้งค่าแล้วเท่านั้น
#   - ผลลัพธ์ : คะแนนและอันดับ หรือข้อความแจ้งว่าข้ามการส่งผล
# ============================================================================

# ตรวจเงื่อนไข: NAME และ JOIN_CODE ต้องไม่เป็นข้อความว่างทั้งคู่ (and = "และ")
if NAME and JOIN_CODE:
    # ส่งไฟล์ผลการถอดความไปยังตารางอันดับ ด้วยฟังก์ชัน submit_to_leaderboard(...) จากเซลล์การตั้งค่า
    #   - "lab06_submission.csv" : ชื่อไฟล์ที่จะส่ง
    submit_to_leaderboard("lab06_submission.csv")
# else: กรณีที่ยังไม่ได้กรอก NAME หรือ JOIN_CODE
else:
    # แสดงข้อความแจ้งว่าข้ามการส่งผล พร้อมวิธีแก้ไข
    print("ข้ามการส่งผลไปยังตารางอันดับ - โปรดระบุ NAME และ JOIN_CODE ในเซลล์การตั้งค่าและรันเซลล์นั้นใหม่หากต้องการส่งผล")

### แนวทางการปรับปรุงให้ดีกว่า baseline

- **โมเดลขนาดใหญ่ขึ้น:** `"small"` ให้ผลดีกว่า `"base"` อย่างชัดเจน ส่วน `"medium"` หรือ `"turbo"` สามารถรันบน GPU รุ่น T4 ได้ แต่ช้ากว่า
- **การชี้นำคำศัพท์ (vocabulary priming):** `initial_prompt="A patient describes symptoms such as headache, cough, phlegm, dizziness, rash, anemia, ligament."` ช่วยให้ Whisper ถอดศัพท์ทางการแพทย์ได้ดีขึ้น (prompt ที่ยาวหรือเฉพาะเจาะจงเกินไปอาจทำให้ผลแย่ลง จึงควรวัดผลทุกครั้ง)
- **การถอดรหัส (decoding):** `beam_size=5` (และ `best_of=5`) ค้นหาข้อความที่เป็นไปได้หลายรูปแบบมากขึ้น ส่วน `temperature=0` ทำให้ผลลัพธ์คงที่ทุกครั้ง
- **เครื่องมืออื่น:** Whisper pipeline ของ Hugging Face `transformers` หรือ `faster-whisper` รองรับการประมวลผลเป็นชุด (batching) และ checkpoint รุ่นใหม่กว่า
- **การตรวจสอบอย่างถูกต้อง:** ข้อความอ้างอิงของชุดทดสอบถูกซ่อนไว้ จึงควรสร้างชุดข้อมูลสำหรับพัฒนา (dev set) ขนาดเล็กโดยถอดความบางคลิปด้วยตนเอง (ส่วนที่ 2) แทนการปรับจูนกับตารางอันดับ ทั้งนี้ ตัวให้คะแนนแปลงเป็นตัวพิมพ์เล็กและตัดเครื่องหมายวรรคตอน แต่ "2" และ "two" ยังนับเป็นคำที่ต่างกัน

In [ ]:
# 🏆 ระบบที่ปรับปรุงแล้ว (ไม่บังคับ) - ปรับค่าที่กำหนด รันเซลล์ และรันเซลล์ส่งผลด้านบนอีกครั้ง
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้ (ส่วนท้าทาย ไม่บังคับ)
#   ปรับปรุงการถอดความให้ดีกว่า baseline ด้วย 3 วิธี ได้แก่
#   1) ใช้โมเดลขนาดใหญ่ขึ้น (MY_MODEL_SIZE)
#   2) การชี้นำคำศัพท์ (vocabulary priming) ด้วยข้อความนำ (MY_PROMPT) ที่มีศัพท์ทางการแพทย์
#   3) ปรับวิธีถอดรหัส (decoding) ด้วย beam search (BEAM_SIZE) และ temperature=0 ให้ผลคงที่
#   แล้วถอดความคลิปทั้ง 30 คลิปใหม่ และบันทึกทับไฟล์ lab06_submission.csv
#   - ผลลัพธ์ : ไฟล์ส่งผลใหม่ และตาราง 5 แถวแรก (จากนั้นรันเซลล์ส่งผลด้านบนอีกครั้ง)
# ============================================================================

# ----------------------------------------------------------------------------
# ส่วนที่ 1: ค่าที่ผู้เรียนปรับได้
# ----------------------------------------------------------------------------
# ขนาดโมเดลที่ต้องการใช้ ("small" ให้ผลดีกว่า "base" ส่วน "medium"/"turbo" ต้องใช้ GPU และช้ากว่า)
MY_MODEL_SIZE = "small"
# ข้อความนำ (prompt) ที่มีศัพท์ทางการแพทย์ ช่วยให้ Whisper คาดเดาคำศัพท์เหล่านี้ได้ดีขึ้น
#   (prompt ที่ยาวหรือเฉพาะเจาะจงเกินไปอาจทำให้ผลแย่ลง จึงควรวัดผลทุกครั้ง)
MY_PROMPT     = "A patient describes symptoms such as headache, cough, phlegm, dizziness, rash, anemia, ligament."
# จำนวนทางเลือกที่ beam search พิจารณาพร้อมกันในการเลือกคำแต่ละขั้น (มากขึ้น = ค้นได้กว้างขึ้นแต่ช้าลง)
BEAM_SIZE     = 5

# ----------------------------------------------------------------------------
# ส่วนที่ 2: เตรียมโมเดล (ใช้ซ้ำหากเคยโหลดไว้แล้ว)
# ----------------------------------------------------------------------------
# พยายามหยิบโมเดลที่เคยโหลดไว้ในแบบฝึกหัดที่ 2 มาใช้ซ้ำ
#   - "models" in globals() : ตรวจว่ามีตัวแปร models อยู่แล้วหรือไม่ (globals() คือรายการตัวแปรทั้งหมดในโน้ตบุ๊ก)
#   - models.get(MY_MODEL_SIZE) : ดึงโมเดลขนาดที่ต้องการจาก dict หากไม่มีจะได้ None (ไม่เกิดข้อผิดพลาด)
#   - ... if ... else None : หากยังไม่มีตัวแปร models ให้ better_model เป็น None
better_model = models.get(MY_MODEL_SIZE) if "models" in globals() else None
# หาก better_model ยังเป็น None (ไม่มีโมเดลที่โหลดไว้) ให้โหลดใหม่ด้วย whisper.load_model(MY_MODEL_SIZE)
#   - A or B : หาก A มีค่าใช้ได้ จะใช้ A มิฉะนั้นจะใช้ B
better_model = better_model or whisper.load_model(MY_MODEL_SIZE)

# ----------------------------------------------------------------------------
# ส่วนที่ 3: ถอดความทุกคลิปด้วยการตั้งค่าที่ปรับปรุงแล้ว
# ----------------------------------------------------------------------------
# สร้าง list ว่างสำหรับสะสมผลทีละคลิป
rows = []
# วนซ้ำทีละคลิป จับคู่รหัสคลิป (cid) กับชื่อไฟล์ (f) ด้วย zip(...)
for cid, f in zip(manifest["id"], manifest["file"]):
    # ดาวน์โหลดคลิปด้วย lab_data(...) แล้วถอดความด้วย better_model.transcribe(...) เก็บผลไว้ใน out
    #   - fp16=USE_FP16            : ใช้ half precision เมื่อมี GPU
    #   - language="en"            : ระบุว่าเป็นภาษาอังกฤษ
    #   - initial_prompt=MY_PROMPT : ข้อความนำที่ช่วยชี้นำคำศัพท์ทางการแพทย์
    #   - beam_size=BEAM_SIZE      : ใช้ beam search ตามจำนวนทางเลือกที่กำหนด
    #   - temperature=0.0          : เลือกคำที่น่าจะเป็นที่สุดเสมอ ไม่สุ่ม ทำให้ผลคงที่ทุกครั้งที่รัน
    out = better_model.transcribe(lab_data(f"asr/{f}"), fp16=USE_FP16, language="en",
                                  initial_prompt=MY_PROMPT, beam_size=BEAM_SIZE, temperature=0.0)
    # เพิ่มผลของคลิปนี้ต่อท้าย rows ในรูป dict {"id": รหัสคลิป, "transcript": ข้อความที่ตัดช่องว่างหัว-ท้ายแล้ว}
    rows.append({"id": cid, "transcript": out["text"].strip()})
# แปลง list ของ dict เป็นตาราง (DataFrame)
sub = pd.DataFrame(rows)
# บันทึกทับไฟล์ส่งผลเดิม (index=False = ไม่บันทึกเลขลำดับแถว)
sub.to_csv("lab06_submission.csv", index=False)
# แสดงจำนวนคลิปและขนาดโมเดลที่ใช้
print("ถอดความแล้ว", len(sub), "คลิปด้วยโมเดล", MY_MODEL_SIZE, "-> lab06_submission.csv")
# แสดงตาราง 5 แถวแรกของผลการถอดความ
display(sub.head())

## ชิ้นงานที่ต้องส่ง (งานเดี่ยว)

ให้ส่งโน้ตบุ๊กนี้ที่ **รันครบทุกเซลล์ตามลำดับจากบนลงล่าง** ผ่านช่องทางการส่งงานของรายวิชา โดยประกอบด้วย

1. ค่า WER ของคลิปตัวอย่าง (ส่วนที่ 1) และตารางเปรียบเทียบขนาดโมเดล (ส่วนที่ 2) ปรากฏในผลลัพธ์
2. คำตอบของ ✍️ **คำถามที่ 1–4** ซึ่งรวมถึงคำอธิบาย 2–3 ประโยคเกี่ยวกับความเสี่ยงที่สำคัญที่สุดของความผิดพลาดของ ASR ในการบันทึกเวชระเบียน (โดยเฉพาะขนาดยาและการปฏิเสธ เช่น "no allergies" เทียบกับ "allergies")

*(ไม่บังคับ)* ผู้เรียนที่เข้าร่วมโจทย์ท้าทายบนตารางอันดับ 🏆 สามารถระบุชื่อที่ใช้และค่า WER ที่ดีที่สุดไว้ในส่วนนี้ได้ โดยไม่มีผลต่อการประเมินชิ้นงานที่ต้องส่ง

## การศึกษาเพิ่มเติม
- ศึกษา [Hugging Face `transformers` Whisper](https://huggingface.co/openai/whisper-small) สำหรับการประมวลผลเป็นชุด (batching) และการปรับจูน (fine-tuning) ให้เหมาะกับสำเนียงในพื้นที่
- ทดลองใช้ Whisper กับเสียงพูดภาษาไทย (`language="th"`) และพิจารณาความถูกต้องในการถอดชื่อยาและตัวเลขภาษาไทย